# Data build & eval harness — Mozilla *Lost in Transcription* (Indonesian–Javanese)

**Purpose.** Build the four artifacts every later notebook depends on, and nothing else.
No model is trained here. Accelerator: **None** — this is a CPU job on the critical path.

| artifact | what it is |
|---|---|
| `segments/*.flac` | one 16 kHz mono clip per Jember segment |
| `manifest.csv` | one row per clip: bounds, three text views, origin counts, split, flags |
| `split.csv` | recording → `train`/`dev` + a 5-fold column. **Frozen after this run.** |
| `lexicon_v1.csv` | word → `jv`/`id`/`both`/`unk`. **Frozen after this run.** |
| `scorer.py` | the official normalizer plus WER, WER-by-origin and an error table |

**Three phases.** One switch, `WRITE_AUDIO`, in the RUN MODE block at the top of cell 1:

| phase | `WRITE_AUDIO` | time | output |
|---|---|---|---|
| 1 — review | `False` *(default)* | ~15 min | **a few MB** — download whole, read the numbers |
| 2 — produce | `True` | ~30 min | ~630 MB — published server-side, never downloaded |
| 3 — verify | — | ~2 min | `02-verify.ipynb`, run against the published dataset |

Phase 1 answers "are the numbers right", phase 2 builds the artifact, phase 3 proves the
artifact that shipped is the one that was reviewed.

There is deliberately **no separate smoke phase**: phase 1 runs this exact code path over
the full corpus, so a smoke run tests nothing extra — it would only be a faster way to hit
the same error. `DEBUG_N_RECORDINGS` exists as a debugging knob for when you are actively
chasing a crash and want a one-minute loop; it is not a step in the workflow. On a GPU
notebook, where a full run costs hours of metered quota, that calculation reverses and a
smoke run *is* worth its own step.

Both passes decode all 208 mp3s — the boundary probe and the true file durations need
that — so pass 1 is only slightly faster. Seed and rules are identical, so `manifest.csv`,
`split.csv` and `lexicon_v1.csv` come out the same in both. One notebook rather than two,
deliberately: two would drift, and the artifacts reviewed would stop being the ones shipped.

**How to run.**
1. Attach `jember-javanese-corpus` (required), `common-voice-corpus` and
   `central-javanese-corpus` (both optional — they only feed the lexicon; without
   them every word falls back to origin `unk`).
2. Accelerator **None**, Internet **off**. Nothing here needs either.
3. **Run All as shipped** — that is phase 1, ~15 minutes. Output tab → Download all (it is
   small), unzip onto the repo root, and read the numbers — especially the boundary probe,
   which decides `PAD_MODE`.
4. Set `WRITE_AUDIO = True`, Run All again. That is phase 2. Download only
   `02-data-build-lite.zip`, and publish the audio server-side with **New Dataset** on the
   Output tab, named `jember-segments-16k`, so the 630 MB never crosses your connection.
5. Run `02-verify.ipynb` against that dataset. That is phase 3, and the gate for
   notebook 03.

**Output convention.** Everything is written to `/kaggle/working/findings/<NOTEBOOK_ID>/`,
which mirrors `<repo>/findings/<NOTEBOOK_ID>/` exactly. Console output is teed to
`run-log.txt`, so there is nothing to copy by hand.

**Corrected 2026-08-25.** The organiser's pre-scoring normalizer **does lowercase**
the first letter of the transcript and the first letter after `. ! ? —` (acronym
exception aside), and **does strip periods** except an ellipsis attached to the end of
a word. Because it runs on both hypothesis and reference before scoring, training text
does not need periods or sentence-initial capitals to be "right" - see `scorer.py`'s
`official_normalize()`. Cell 9 prices what a *lowercased-and-cleaned* hypothesis costs;
that number predates this correction and should be re-read against the fixed scorer.

## 0 — Dependencies

`soundfile` and `librosa` ship with the Kaggle image. `jiwer` is optional: `scorer.py`
carries its own aligner and only uses jiwer to cross-check the totals when it is present.

In [ ]:
!pip list 2>/dev/null | grep -Ei '^(librosa|soundfile|jiwer|pandas|numpy) ' || echo 'listing skipped'

## 1 — Setup, output convention, path discovery

Resolves the corpora by marker file rather than by slug or depth: this account mounts
inputs owner-scoped (`/kaggle/input/datasets/<user>/<slug>/…`), so any hardcoded path
would break the moment a dataset is renamed. Read the RESOLVED block before continuing.

In [ ]:
# ============================================================
# CELL 1 - SETUP, OUTPUT CONVENTION, PATH DISCOVERY
# ============================================================
import os, sys, re, csv, json, math, time, random, shutil, unicodedata, collections
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

# ============================================================
# RUN MODE - three phases, one switch.
#
#   phase 1  review : WRITE_AUDIO = False   ~15 min, a few MB - download and read
#   phase 2  produce: WRITE_AUDIO = True    ~30 min, ~630 MB   - published, not downloaded
#   phase 3         : notebooks/02-verify.ipynb, against the published dataset
# ============================================================

# Write the ~630 MB of clips? False computes every number and writes none of
# them. See the header of this notebook for the reasoning.
WRITE_AUDIO = False

# DEBUGGING KNOB, NOT A PHASE. An int builds everything from the first N
# recordings, in about a minute. Phase 1 already exercises this exact code path
# on the full corpus, so this buys nothing except a fast loop while you are
# actively chasing a crash. Leave it None for every normal run.
# Its output goes to findings/<id>-dryrun/, so a partial build can never be
# unzipped over the real artifacts by accident.
DEBUG_N_RECORDINGS = None

# One seed for the whole notebook. It is written into dataset_findings.json,
# because a split nobody can reproduce is not a frozen split.
SEED = 20260825
random.seed(SEED); np.random.seed(SEED)

# Vocabulary tokenizer: letters (incl. diacritics) with internal - and '.
# NOT the scoring tokenizer - scoring splits on whitespace, see scorer.tokenize.
RE_TOKEN = r"[^\W\d_]+(?:[-'][^\W\d_]+)*"

FINDINGS = {}
# NOTE: this reset is why partial re-runs do not work. Re-running THIS cell
# discards everything the later cells collected. Always Run All.

def rec(section, **kw):
    FINDINGS.setdefault(section, {}).update(kw)

def head(title):
    print("\n" + "=" * 74); print(title); print("=" * 74)

def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return round(float(o), 4)
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float):          return round(o, 4)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

def desc(a, name=""):
    # Compact distribution summary used all over this notebook.
    a = np.asarray([x for x in a if x is not None and not (isinstance(x, float) and math.isnan(x))],
                   dtype=float)
    if a.size == 0:
        return dict(n=0)
    q = lambda p: round(float(np.percentile(a, p)), 3)
    d = dict(n=int(a.size), mean=round(float(a.mean()), 3), min_s=q(0), p05=q(5), p25=q(25),
             median=q(50), p75=q(75), p95=q(95), max_s=q(100), total_h=round(float(a.sum()) / 3600, 3))
    if name:
        print("%-22s n=%-6d min=%.2f p25=%.2f med=%.2f p75=%.2f p95=%.2f max=%.2f  sum=%.2f h"
              % (name, d["n"], d["min_s"], d["p25"], d["median"], d["p75"], d["p95"],
                 d["max_s"], d["total_h"]))
    return d

# ============================================================
# OUTPUT CONVENTION - mirrors the local repo layout
# ------------------------------------------------------------
# Kaggle writes to /kaggle/working. Everything this notebook produces goes to
#     /kaggle/working/findings/<NOTEBOOK_ID>/
# which is the same path as
#     <repo>/findings/<NOTEBOOK_ID>/
# so the downloaded Output archive can be dropped straight onto the repo root
# and every file lands where it belongs. Copy this block into every new
# notebook and change NOTEBOOK_ID only.
# ============================================================
NOTEBOOK_ID  = "02-data-build"           # must match notebooks/<NOTEBOOK_ID>.ipynb
WORKING      = Path("/kaggle/working")
# A smoke run writes to findings/<id>-dryrun/ instead. Deliberate break from
# the mirroring rule: without it, unzipping a smoke run onto the repo would
# quietly overwrite good artifacts with output built from five recordings.
RUN_TAG      = "-dryrun" if DEBUG_N_RECORDINGS else ""
OUT_DIR      = WORKING / "findings" / (NOTEBOOK_ID + RUN_TAG)
ARTIFACT_DIR = OUT_DIR / "_kaggle-artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def out(filename):
    # Every write in this notebook goes through here. Nothing lands loose.
    p = OUT_DIR / filename
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

# The audio lives inside the same mirrored tree, so the download archive is
# self-contained and recordings/ can be published as a Kaggle dataset as-is.
# ONE FLAC PER RECORDING, not per segment: 208 files instead of 6.638. Kaggle's
# dataset builder truncated the per-segment version at 500 files, and thousands
# of tiny files make every mount slow even where it does work. Segments are
# addressed by sample offset in the manifest - see cell 7.
AUDIO_DIR = out("recordings")
META_DIR  = out("recordings/_meta")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Tee stdout to a log file so the console output is saved without hand-copying.
# NOTE: in Jupyter the live stream is sys.stdout, not sys.__stdout__ - writing to
# the latter would send the output to the kernel process instead of the cell.
_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

LOG_PATH = out("run-log.txt")
try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
phase = ("DEBUG" if DEBUG_N_RECORDINGS else ("2 PRODUCE" if WRITE_AUDIO else "1 REVIEW"))
print("=" * 74)
print("RUN PHASE  : %s      DEBUG_N_RECORDINGS = %s   WRITE_AUDIO = %s"
      % (phase, DEBUG_N_RECORDINGS, WRITE_AUDIO))
print("=" * 74)
if DEBUG_N_RECORDINGS:
    print("!! PARTIAL BUILD from %d recordings - debugging only." % DEBUG_N_RECORDINGS)
    print("!! Output goes to findings/%s%s/ so it can never be confused"
          % (NOTEBOOK_ID, RUN_TAG))
    print("!! with the real artifacts. Nothing here is publishable.")
print("output dir : %s" % OUT_DIR)
print("console log: %s  (restarts every time this cell runs)" % LOG_PATH)
print("seed       : %d" % SEED)

INPUT = Path("/kaggle/input")

# ------------------------------------------------------------------
# Discovery is CONTENT-BASED and DEPTH-AGNOSTIC. This account mounts
# owner-scoped (/kaggle/input/datasets/<user>/<slug>/...), so a corpus is
# found by a file only it has and its root derived from that hit.
# ------------------------------------------------------------------
head("DISCOVERY")

def shallow(base, max_depth=4):
    rows = []
    base = Path(base)
    for dp, dns, fns in os.walk(base):
        d = Path(dp)
        depth = len(d.relative_to(base).parts)
        if depth > max_depth:
            dns[:] = []; continue
        if depth >= 1:
            rows.append((depth, d, len(fns)))
        if len(fns) > 200:      # a clips/ dir - do not descend further
            dns[:] = []
    return rows

layout = shallow(INPUT) if INPUT.exists() else []
for depth, d, nf in layout[:40]:
    print("  %s%s%s" % ("    " * (depth - 1), d.name, ("  [%d files]" % nf) if nf else "/"))
if len(layout) > 40:
    print("  ... +%d more directories" % (len(layout) - 40))

# 1) Common Voice - the only corpus shipping clip_durations.tsv
cv_hit = next(iter(sorted(INPUT.glob("**/clip_durations.tsv"))), None)
CV_DIR = cv_hit.parent if cv_hit else None

# 2) Jember - a TSV whose header carries start / end / text
jem_hits = []
for q in INPUT.glob("**/*.tsv"):
    if CV_DIR is not None and q.parent == CV_DIR:
        continue
    try:
        hdr = q.open(encoding="utf-8", errors="replace").readline().lower()
    except Exception:
        continue
    if "start" in hdr and "end" in hdr and "text" in hdr:
        jem_hits.append(q)
JEMBER_TSV = max(jem_hits, key=lambda q: q.stat().st_size) if jem_hits else None
JEMBER_DS = JEMBER_TSV.parent if JEMBER_TSV else None

# 2b) the mp3 directory: whichever directory under the Jember root holds mp3s
JEMBER_AUDIO = None
if JEMBER_DS is not None:
    cand = collections.Counter()
    for q in JEMBER_DS.glob("**/*.mp3"):
        cand[q.parent] += 1
    if cand:
        JEMBER_AUDIO = cand.most_common(1)[0][0]

# 3) Central Javanese TTS - the corpus whose clips ship a per-folder mapping.tsv
TTS_MAPPINGS = sorted(INPUT.glob("**/mapping.tsv"))
TTS_DS = None
if TTS_MAPPINGS:
    TTS_DS = Path(os.path.commonpath([str(p.parent) for p in TTS_MAPPINGS]))

# 4) Common Voice sentence pool. validated_sentences.tsv is the sentence table;
#    validated.tsv is the clip table and carries the same text in `sentence`.
CV_SENT = None
if CV_DIR is not None:
    for name in ("validated_sentences.tsv", "validated.tsv"):
        if (CV_DIR / name).exists():
            CV_SENT = CV_DIR / name; break

# ==================================================================
# MANUAL OVERRIDE - only if a line below says NOT FOUND. Copy the real
# path from the layout listing at the top of this cell.
# ==================================================================
# JEMBER_TSV   = Path("/kaggle/input/.../Jember ... - 1-200.tsv")
# JEMBER_AUDIO = Path("/kaggle/input/.../mp3 audio")
# CV_SENT      = Path("/kaggle/input/.../id/validated_sentences.tsv")
# TTS_MAPPINGS = sorted(Path("/kaggle/input/.../RECORDING TTS").glob("**/mapping.tsv"))

print("")
print("RESOLVED:")
print("  JEMBER_TSV   =", JEMBER_TSV or "NOT FOUND")
print("  JEMBER_AUDIO =", JEMBER_AUDIO or "NOT FOUND",
      "" if JEMBER_AUDIO is None else "(%d mp3)" % len(list(JEMBER_AUDIO.glob("*.mp3"))))
print("  CV_SENT      =", CV_SENT or "NOT FOUND")
print("  TTS mappings =", "%d files under %s" % (len(TTS_MAPPINGS), TTS_DS) if TTS_MAPPINGS else "NOT FOUND")

missing = [n for n, v in [("JEMBER_TSV", JEMBER_TSV), ("JEMBER_AUDIO", JEMBER_AUDIO)] if v is None]
if missing:
    print("")
    print("!!! UNRESOLVED, and these two are mandatory: %s" % missing)
    print("!!! Fill the MANUAL OVERRIDE block above and re-run this cell.")
else:
    print("")
    print("Jember resolved - safe to run the rest. CV/TTS only feed the lexicon;")
    print("without them every word falls back to origin 'unk'.")

rec("env",
    notebook=NOTEBOOK_ID, seed=SEED, python=sys.version.split()[0], pandas=pd.__version__,
    numpy=np.__version__, jember_tsv=str(JEMBER_TSV), jember_audio=str(JEMBER_AUDIO),
    cv_sentences=str(CV_SENT), n_tts_mappings=len(TTS_MAPPINGS), unresolved=missing,
    kaggle_layout=[str(d.relative_to(INPUT)) for _, d, _ in layout[:30]])

## 2 — `scorer.py`

Written to disk first, then imported, so the notebook uses exactly the file that ships to
notebooks 03–07. The module is stdlib-only; `jiwer` and `pandas` are optional niceties.

`wer_by_origin` attributes substitutions and deletions to the origin class of the
**reference** token. Insertions have no reference token, so they are reported globally and
as a frequency list, never spread across the classes — that would invent an attribution
the alignment does not contain.

In [ ]:
# ============================================================
# CELL 2 - THE SCORER, WRITTEN TO DISK AND THEN IMPORTED
# ============================================================
SCORER_SRC = r'''# scorer.py - evaluation harness for Mozilla "Lost in Transcription" (id-jv).
#
# Written by notebooks/02-data-build.ipynb. Do not edit this file in the output
# tree; edit the SCORER_SRC string in the notebook and re-run, otherwise the
# module and the manifest it produced drift apart.
#
# Dependencies: stdlib only. jiwer is used ONLY to cross-check the totals when
# it happens to be installed; pandas ONLY so error_table can hand back a
# DataFrame. Neither is required.
#
# ---------------------------------------------------------------------------
# Corrected 2026-08-25 against the live Problem description page (the first
# published list, read 23 Aug, was incomplete). The organiser's normalizer:
#   * DOES lowercase  the first letter of the transcript and the first letter
#     after . ! ? or em-dash (acronym exception: left alone if the next char
#     is also a capital). Mid-sentence case is otherwise compared as written.
#   * DOES strip periods, including the sentence-final one - EXCEPT an
#     ellipsis attached to the end of a word (disfluency marker), which is
#     kept verbatim. A free-standing "..." (space on both sides) is removed
#     entirely, same as before.
#   * still does not strip hyphens -> "arek-arek" stays one token
# Because both hypothesis and reference go through this same normalizer
# before scoring, training text does not need periods or sentence-initial
# capitals to be "right" - they are compared post-normalization either way.
# ---------------------------------------------------------------------------

import re
import csv
import random
import unicodedata
import collections

__all__ = [
    "official_normalize", "fold_diacritics", "lex_key", "tokenize",
    "align_words", "wer", "wer_by_origin", "error_table",
    "wer_ci", "compare_systems",
    "load_lexicon", "self_test",
    "PUNCT_REMOVED", "ORIGINS",
]

VERSION = "1.0"

# The complete list of characters the organiser removes. Nothing may be added
# here without a rule change in writing: every extra character is a form the
# model would learn to drop and then be scored wrong for dropping.
PUNCT_REMOVED = ',?¿¡!";:'

ORIGINS = ("jv", "id", "both", "unk")

_RE_BRACKET_TAG = re.compile(r"\[[^\]]*\]")          # [laugh], [???], [cough]
_RE_PAREN_UNK = re.compile(r"\(\s*\?+\s*\)")         # (???) inaudible marker
_RE_SENT_BOUNDARY = re.compile(r"[.!?\u2014]")       # . ! ? em-dash
# Standalone ellipsis = surrounded by whitespace or string edges: removed.
_RE_ELLIPSIS_FREE = re.compile(r"(?<!\S)\.{3,}(?!\S)")
# Attached ellipsis = glued to the end of a word: kept verbatim, protected
# from the generic period-strip step below.
_RE_ELLIPSIS_ATTACHED = re.compile(r"(?<=\S)\.{3,}")
_RE_PUNCT = re.compile("[" + re.escape(PUNCT_REMOVED) + "]")
_RE_WS = re.compile(r"\s+")
_RE_EDGE_NONWORD = re.compile(r"^\W+|\W+$", re.UNICODE)


_SKIP_TO_LETTER = frozenset(PUNCT_REMOVED)  # chars this step must see past


def _next_letter_pos(chars, start):
    # Skip whitespace AND anything the punctuation-strip step removes later
    # (comma, quote, ...). Without the latter, "e..., \"Kapan" finds the
    # quote, not the K, on this pass but finds K once the quote is already
    # gone on a second pass - non-idempotent AND wrong on the first pass.
    n = len(chars)
    j = start
    while j < n and (chars[j].isspace() or chars[j] in _SKIP_TO_LETTER):
        j += 1
    return j


def _lowercase_sentence_initial(s):
    # Lowercase the first letter of the string, and the first letter after
    # each . ! ? or em-dash. Skip a letter immediately followed by another
    # capital - that is the organiser's acronym exception.
    chars = list(s)
    n = len(chars)
    positions = [_next_letter_pos(chars, 0)] if n else []
    for i, c in enumerate(chars):
        if _RE_SENT_BOUNDARY.match(c):
            j = _next_letter_pos(chars, i + 1)
            if j < n:
                positions.append(j)
    for i in positions:
        if i >= n:
            continue
        c = chars[i]
        if c.isupper():
            nxt = chars[i + 1] if i + 1 < n else ""
            if not (nxt and nxt.isupper()):
                chars[i] = c.lower()
    return "".join(chars)


def _strip_periods_keep_attached_ellipsis(s):
    # Protect word-attached ellipses so the blanket period-strip below
    # cannot touch them, then remove every other period, including a
    # standalone "..." (handled the same as before) and any lone ".".
    stash = []

    def _protect(m):
        stash.append(m.group(0))
        return "\x00%d\x00" % (len(stash) - 1)

    s = _RE_ELLIPSIS_ATTACHED.sub(_protect, s)
    s = _RE_ELLIPSIS_FREE.sub(" ", s)
    s = s.replace(".", "")
    for i, original in enumerate(stash):
        s = s.replace("\x00%d\x00" % i, original)
    return s


def official_normalize(text):
    # Rule order follows the organiser's published list: tags/unintelligible
    # markers first, then sentence-initial lowercasing (needs the . ! ? / em
    # dash delimiters still in place), then punctuation, then periods.
    if text is None:
        return ""
    s = str(text)
    s = _RE_BRACKET_TAG.sub(" ", s)
    s = _RE_PAREN_UNK.sub(" ", s)
    s = _lowercase_sentence_initial(s)
    s = _RE_PUNCT.sub("", s)
    s = _strip_periods_keep_attached_ellipsis(s)
    return _RE_WS.sub(" ", s).strip()


def fold_diacritics(text):
    # e-circumflex / e-grave / e-caron / e-acute -> e, d-with-dot-below -> d.
    # Kept as a SEPARATE view of the text, never as a replacement: whether the
    # organiser folds diacritics before scoring is still an open question.
    s = unicodedata.normalize("NFD", "" if text is None else str(text))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return unicodedata.normalize("NFC", s)


def lex_key(word):
    # Lookup key for the origin lexicon: diacritics folded, lowercased, edge
    # punctuation stripped. "Ndek," "ndek." and "ndek" must hit one entry.
    w = fold_diacritics(word).lower()
    return _RE_EDGE_NONWORD.sub("", w)


def tokenize(text, normalize=True):
    # WER is a per-word metric over the normalizer's output, so the split is
    # plain whitespace. Do NOT use a letters-only tokenizer here - that would
    # silently drop the trailing periods the organiser keeps.
    s = official_normalize(text) if normalize else ("" if text is None else str(text))
    return s.split()


# ---------------------------------------------------------------------------
# alignment
# ---------------------------------------------------------------------------

HIT, SUB, DEL, INS = "hit", "sub", "del", "ins"


def align_words(ref, hyp):
    # Levenshtein backtrace over word lists. Returns [(op, ref_idx, hyp_idx)]
    # in reading order; ref_idx is None for insertions, hyp_idx None for
    # deletions. Ties break hit > sub > del > ins so the attribution in
    # wer_by_origin is reproducible across runs and machines.
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        d[i][0] = i
    for j in range(1, m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        cur, prev, ri = d[i], d[i - 1], ref[i - 1]
        for j in range(1, m + 1):
            c = 0 if ri == hyp[j - 1] else 1
            cur[j] = min(prev[j - 1] + c, prev[j] + 1, cur[j - 1] + 1)
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1] and d[i][j] == d[i - 1][j - 1]:
            ops.append((HIT, i - 1, j - 1)); i -= 1; j -= 1
        elif i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + 1:
            ops.append((SUB, i - 1, j - 1)); i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append((DEL, i - 1, None)); i -= 1
        else:
            ops.append((INS, None, j - 1)); j -= 1
    ops.reverse()
    return ops


def _pairs(refs, hyps):
    if isinstance(refs, str) or isinstance(hyps, str):
        raise TypeError("wer() takes sequences of strings, not a single string")
    refs = list(refs)
    hyps = list(hyps)
    if len(refs) != len(hyps):
        raise ValueError("refs and hyps differ in length: %d vs %d" % (len(refs), len(hyps)))
    return refs, hyps


# ---------------------------------------------------------------------------
# metrics
# ---------------------------------------------------------------------------

def wer(refs, hyps, normalize=True):
    # Corpus-level WER: one pooled ratio, not the mean of per-utterance ratios.
    # A per-utterance mean lets a 2-word utterance outweigh a 30-word one.
    refs, hyps = _pairs(refs, hyps)
    n_hit = n_sub = n_del = n_ins = n_ref = n_hyp = 0
    for r, h in zip(refs, hyps):
        rt = tokenize(r, normalize)
        ht = tokenize(h, normalize)
        n_ref += len(rt)
        n_hyp += len(ht)
        for op, _, _ in align_words(rt, ht):
            if op == HIT:
                n_hit += 1
            elif op == SUB:
                n_sub += 1
            elif op == DEL:
                n_del += 1
            else:
                n_ins += 1
    errs = n_sub + n_del + n_ins
    den = float(n_ref) if n_ref else 1.0
    return dict(
        wer=errs / den if n_ref else 0.0,
        sub_rate=n_sub / den, del_rate=n_del / den, ins_rate=n_ins / den,
        n_utts=len(refs), n_ref_tokens=n_ref, n_hyp_tokens=n_hyp,
        n_hit=n_hit, n_sub=n_sub, n_del=n_del, n_ins=n_ins, n_errors=errs,
    )


def _per_utterance(refs, hyps, normalize=True):
    # (n_errors, n_ref_tokens) per utterance - the unit both bootstraps resample.
    rows = []
    for r, h in zip(refs, hyps):
        rt = tokenize(r, normalize)
        ht = tokenize(h, normalize)
        e = sum(1 for op, _, _ in align_words(rt, ht) if op != HIT)
        rows.append((e, len(rt)))
    return rows


def wer_ci(refs, hyps, n_boot=1000, alpha=0.05, seed=0, normalize=True):
    # Bootstrap confidence interval for WER, resampling UTTERANCES, not tokens.
    # Tokens inside one utterance fail together - a model that loses the thread
    # gets the whole sentence wrong - so treating tokens as independent would
    # report an interval several times too narrow.
    #
    # This is the honest answer to "is this difference real?", and it costs no
    # GPU at all. Reach for it before reaching for a second training run.
    refs, hyps = _pairs(refs, hyps)
    rows = _per_utterance(refs, hyps, normalize)
    n = len(rows)
    tot_e = sum(e for e, _ in rows)
    tot_r = sum(r for _, r in rows)
    point = tot_e / float(tot_r) if tot_r else 0.0
    if n == 0 or n_boot <= 0:
        return dict(wer=point, lo=point, hi=point, se=0.0, n_utts=n, n_boot=0)
    rng = random.Random(seed)
    draws = []
    for _ in range(n_boot):
        e = r = 0
        for _ in range(n):
            de, dr = rows[rng.randrange(n)]
            e += de; r += dr
        draws.append(e / float(r) if r else 0.0)
    draws.sort()
    lo = draws[max(0, int((alpha / 2.0) * n_boot) - 1)]
    hi = draws[min(n_boot - 1, int((1 - alpha / 2.0) * n_boot))]
    mean = sum(draws) / len(draws)
    se = (sum((d - mean) ** 2 for d in draws) / max(len(draws) - 1, 1)) ** 0.5
    return dict(wer=point, lo=lo, hi=hi, se=se, n_utts=n, n_boot=n_boot,
                half_width=(hi - lo) / 2.0)


def compare_systems(refs, hyps_a, hyps_b, n_boot=1000, seed=0, normalize=True):
    # PAIRED bootstrap: both systems are scored on the same resampled
    # utterances, so the shared difficulty of those utterances cancels out.
    # Far more sensitive than asking whether two separate CIs overlap - two
    # systems can have overlapping intervals and still differ reliably.
    #
    # p_b_better is the share of resamples in which B beats A. Read it as
    # evidence, not as a ritual: 0.5 means the data cannot tell them apart.
    refs, _ = _pairs(refs, hyps_a)
    _pairs(refs, hyps_b)
    ra = _per_utterance(refs, hyps_a, normalize)
    rb = _per_utterance(refs, hyps_b, normalize)
    n = len(ra)
    wa = sum(e for e, _ in ra) / float(max(sum(r for _, r in ra), 1))
    wb = sum(e for e, _ in rb) / float(max(sum(r for _, r in rb), 1))
    rng = random.Random(seed)
    deltas = []
    wins = 0
    for _ in range(n_boot):
        ea = ra_ = eb = rb_ = 0
        for _ in range(n):
            i = rng.randrange(n)
            ea += ra[i][0]; ra_ += ra[i][1]
            eb += rb[i][0]; rb_ += rb[i][1]
        d = (eb / float(rb_ if rb_ else 1)) - (ea / float(ra_ if ra_ else 1))
        deltas.append(d)
        if d < 0:
            wins += 1
    deltas.sort()
    return dict(wer_a=wa, wer_b=wb, delta=wb - wa,
                delta_lo=deltas[max(0, int(0.025 * n_boot) - 1)],
                delta_hi=deltas[min(n_boot - 1, int(0.975 * n_boot))],
                p_b_better=wins / float(n_boot), n_utts=n, n_boot=n_boot)


def load_lexicon(path):
    # Reads lexicon_v1.csv (word,origin,source) into {lex_key: origin}.
    lex = {}
    with open(str(path), "r", encoding="utf-8", newline="") as fh:
        for row in csv.DictReader(fh):
            w = (row.get("word") or "").strip()
            o = (row.get("origin") or "unk").strip()
            if w:
                lex[lex_key(w)] = o if o in ORIGINS else "unk"
    return lex


def _as_lexicon(lexicon):
    if lexicon is None:
        return {}
    if isinstance(lexicon, dict):
        return lexicon
    return load_lexicon(lexicon)


def _origin_of(token, lex):
    return lex.get(lex_key(token), "unk")


def _cosmetic_kind(ref_tok, hyp_tok):
    # Which of the free wins is this substitution? Free = removable in
    # post-processing or by a spelling convention, without touching the model.
    kinds = []
    if ref_tok != hyp_tok:
        if ref_tok.lower() == hyp_tok.lower():
            kinds.append("case")
        if fold_diacritics(ref_tok) == fold_diacritics(hyp_tok):
            kinds.append("diacritic")
        a = _RE_EDGE_NONWORD.sub("", ref_tok)
        b = _RE_EDGE_NONWORD.sub("", hyp_tok)
        if a == b and a != "":
            kinds.append("punct")
        if not kinds:
            af = fold_diacritics(a).lower()
            bf = fold_diacritics(b).lower()
            if af == bf and af != "":
                kinds.append("mixed")
    return kinds


def wer_by_origin(refs, hyps, lexicon, normalize=True):
    # Substitutions and deletions are attributed to the origin class of the
    # REFERENCE token. Insertions have no reference token, so they are reported
    # globally and as a frequency list - never spread across the classes, which
    # would invent an attribution the alignment does not contain.
    refs, hyps = _pairs(refs, hyps)
    lex = _as_lexicon(lexicon)
    per = {o: dict(n_ref=0, hit=0, sub=0, dele=0) for o in ORIGINS}
    n_ins = 0
    ins_counter = collections.Counter()
    cosmetic = collections.Counter()
    n_sub_total = 0
    for r, h in zip(refs, hyps):
        rt = tokenize(r, normalize)
        ht = tokenize(h, normalize)
        for op, ri, hi in align_words(rt, ht):
            if op == INS:
                n_ins += 1
                ins_counter[ht[hi]] += 1
                continue
            o = _origin_of(rt[ri], lex)
            if o not in per:
                o = "unk"
            per[o]["n_ref"] += 1
            if op == HIT:
                per[o]["hit"] += 1
            elif op == SUB:
                per[o]["sub"] += 1
                n_sub_total += 1
                for k in (_cosmetic_kind(rt[ri], ht[hi]) or ["real"]):
                    cosmetic[k] += 1
            else:
                per[o]["dele"] += 1
    n_ref = sum(per[o]["n_ref"] for o in ORIGINS)
    n_sub = sum(per[o]["sub"] for o in ORIGINS)
    n_del = sum(per[o]["dele"] for o in ORIGINS)
    den = float(n_ref) if n_ref else 1.0
    res = dict(
        wer_total=(n_sub + n_del + n_ins) / den if n_ref else 0.0,
        sub_rate=n_sub / den, del_rate=n_del / den, ins_rate=n_ins / den,
        n_ref_tokens=n_ref, n_sub=n_sub, n_del=n_del, n_ins=n_ins,
        n_utts=len(refs),
        top_insertions=ins_counter.most_common(25),
    )
    for o in ORIGINS:
        d = per[o]
        dd = float(d["n_ref"]) if d["n_ref"] else 1.0
        # NOTE: a class WER counts only errors carrying a reference token of
        # that class. Comparable across systems, but not a WER in the strict
        # sense - the insertions are missing from the numerator by design.
        res["wer_" + o] = (d["sub"] + d["dele"]) / dd if d["n_ref"] else 0.0
        res["n_ref_" + o] = d["n_ref"]
        res["n_sub_" + o] = d["sub"]
        res["n_del_" + o] = d["dele"]
    for k in ("case", "diacritic", "punct", "mixed", "real"):
        res["sub_" + k] = cosmetic.get(k, 0)
    res["cosmetic_sub_pct"] = round(
        100.0 * (n_sub_total - cosmetic.get("real", 0)) / max(n_sub_total, 1), 2)
    return res


def error_table(refs, hyps, lexicon, top_k=50, normalize=True):
    # Most frequent ref -> hyp confusions, tagged with the three cosmetic
    # classes. Deletions appear with hyp = "", insertions with ref = "".
    refs, hyps = _pairs(refs, hyps)
    lex = _as_lexicon(lexicon)
    cnt = collections.Counter()
    for r, h in zip(refs, hyps):
        rt = tokenize(r, normalize)
        ht = tokenize(h, normalize)
        for op, ri, hi in align_words(rt, ht):
            if op == HIT:
                continue
            a = rt[ri] if ri is not None else ""
            b = ht[hi] if hi is not None else ""
            cnt[(op, a, b)] += 1
    rows = []
    for (op, a, b), n in cnt.most_common(top_k):
        kinds = _cosmetic_kind(a, b) if op == SUB else []
        rows.append(dict(
            op=op, ref=a, hyp=b, count=n,
            ref_origin=_origin_of(a, lex) if a else "",
            diacritic_only=("diacritic" in kinds),
            case_only=("case" in kinds),
            punct_only=("punct" in kinds),
            cosmetic=bool(kinds),
        ))
    cols = ["op", "ref", "hyp", "count", "ref_origin",
            "diacritic_only", "case_only", "punct_only", "cosmetic"]
    try:
        import pandas as pd
        return pd.DataFrame(rows, columns=cols)
    except Exception:
        return rows


# ---------------------------------------------------------------------------
# self test - notebooks 03..07 should call this once after importing
# ---------------------------------------------------------------------------

def self_test(refs=None, verbose=True):
    checks = []

    def ck(name, cond, detail=""):
        checks.append((name, bool(cond), detail))

    n = official_normalize
    ck("lowercases sentence-initial letter", n("Oke Bismillah") == "oke Bismillah")
    ck("keeps mid-sentence case", n("Soale Aku sibuk.") == "soale Aku sibuk")
    ck("leaves acronym alone", n("NASA is here.") == "NASA is here")
    ck("strips sentence-final period", n("Soale aku sibuk.") == "soale aku sibuk")
    ck("keeps hyphen", n("Aja suwi-suwi.") == "aja suwi-suwi")
    ck("drops comma", n("Langsung ae ya, aja suwi.") == "langsung ae ya aja suwi")
    ck("drops question mark", n("Asline wong ngendi ?") == "asline wong ngendi")
    ck("drops bracket tag", n("iya [laugh] ngono") == "iya ngono")
    ck("drops inaudible", n("iku (???) ngono") == "iku ngono")
    ck("drops free ellipsis", n("iku ... ngono") == "iku ngono")
    ck("keeps attached ellipsis", n("mba... e...,") == "mba... e...")
    ck("lowercases after sentence boundary", n("Sek. Terus mangkat.") == "sek terus mangkat")
    ck("sees past a comma to the next letter", n("E..., Sabtu ngajar privat.") == "e... sabtu ngajar privat")
    ck("sees past a quote to the next letter", n('takon, e..., "Kapan nikah?"') == "takon e... kapan nikah")
    ck("collapses whitespace", n("a  \t b") == "a b")
    ck("idempotent on samples", all(n(n(s)) == n(s) for s in [
        "Tapi, keba, e..., keseringan lek ndek kono.",
        "iku ... , ngono", "[a] (???) b...", "Soale aku sibuk.",
        "E..., Sabtu ngajar privat.", 'takon, e..., "Kapan nikah?"']))
    ck("folds diacritics", fold_diacritics("têrus arèk-arèk") == "terus arek-arek")
    ck("lex key", lex_key("Ndèk,") == "ndek")

    r = ["aku mangan sega", "iku ngono ae"]
    ck("wer(refs, refs) == 0", wer(r, r)["wer"] == 0.0)
    ck("wer(refs, empty) == 1", wer(r, [""] * len(r))["wer"] == 1.0)
    ck("counts one sub", wer(["aku mangan sega"], ["aku mangan pitik"])["n_sub"] == 1)
    ck("counts one ins", wer(["aku mangan"], ["aku wis mangan"])["n_ins"] == 1)
    ck("counts one del", wer(["aku wis mangan"], ["aku mangan"])["n_del"] == 1)

    lex = {"aku": "jv", "mangan": "jv", "sega": "jv", "saya": "id", "makan": "id"}
    o = wer_by_origin(["aku mangan sega"], ["aku mangan pitik"], lex)
    ck("origin attribution", o["n_ref_jv"] == 3 and o["n_sub_jv"] == 1)
    ck("insertions unattributed",
       wer_by_origin(["aku mangan"], ["aku wis mangan"], lex)["n_ins"] == 1)
    ck("case-only substitution flagged",  # mid-sentence word: sentence-initial
       wer_by_origin(["wau Terus arek"], ["wau terus arek"], lex)["sub_case"] == 1)
    ck("diacritic-only substitution flagged",
       wer_by_origin(["têrus arek"], ["terus arek"], lex)["sub_diacritic"] == 1)

    ci = wer_ci(r, r, n_boot=50)
    ck("wer_ci on a perfect system is 0 with no spread",
       ci["wer"] == 0.0 and ci["lo"] == 0.0 and ci["hi"] == 0.0)
    ci2 = wer_ci(r, [""] * len(r), n_boot=50)
    ck("wer_ci brackets the point estimate", ci2["lo"] <= ci2["wer"] <= ci2["hi"])
    cmp_ = compare_systems(["aku mangan sega"] * 20,
                           ["aku mangan sega"] * 20, ["aku mangan pitik"] * 20, n_boot=100)
    ck("compare_systems sees a strictly worse B",
       cmp_["delta"] > 0 and cmp_["p_b_better"] == 0.0)

    if refs:
        refs = list(refs)
        ck("idempotent on corpus", all(n(n(s)) == n(s) for s in refs))
        ck("corpus wer(refs, refs) == 0", wer(refs, refs)["wer"] == 0.0)
        ck("corpus wer(refs, empty) == 1", wer(refs, [""] * len(refs))["wer"] == 1.0)
        try:
            import jiwer
            rot = refs[1:] + refs[:1]
            mine = wer(refs, rot)["wer"]
            rn = [official_normalize(x) for x in refs]
            hn = [official_normalize(x) for x in rot]
            keep = [i for i in range(len(rn)) if rn[i].strip()]
            theirs = jiwer.wer([rn[i] for i in keep], [hn[i] for i in keep])
            ck("agrees with jiwer (%.6f vs %.6f)" % (mine, theirs), abs(mine - theirs) < 1e-6)
        except ImportError:
            ck("jiwer cross-check", True, "skipped - jiwer not installed")

    ok = all(c for _, c, _ in checks)
    if verbose:
        for name, cond, detail in checks:
            print("  [%s] %s%s" % ("ok" if cond else "FAIL", name,
                                   ("  (%s)" % detail) if detail else ""))
        print("  scorer self-test: %d/%d passed"
              % (sum(1 for _, c, _ in checks if c), len(checks)))
    return ok, checks


if __name__ == "__main__":
    _ok, _ = self_test()
    raise SystemExit(0 if _ok else 1)
'''

# ---- write it, import it, prove it -------------------------------------
# The module is written to disk FIRST and then imported, so the notebook uses
# exactly the file that ships. A copy that only lives in the kernel is a copy
# that silently differs from the one notebooks 03-07 will load.
out("scorer.py").write_text(SCORER_SRC, encoding="utf-8")
print("wrote %s  (%.1f KB)" % (out("scorer.py"), len(SCORER_SRC) / 1024))

if str(OUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUT_DIR))
import importlib
import scorer as _scorer_mod
scorer = importlib.reload(_scorer_mod)
official_normalize = scorer.official_normalize
fold_diacritics    = scorer.fold_diacritics
lex_key            = scorer.lex_key

head("SCORER SELF-TEST (unit level - the corpus-level checks run in cell 9)")
_ok, _checks = scorer.self_test()
if not _ok:
    raise RuntimeError("scorer.py failed its own unit tests - fix SCORER_SRC before continuing")

head("WHAT THE NORMALIZER DOES TO REAL LINES")
for s in ["Langsung ae ya, aja suwi-suwi.",
          "Tapi, keba, e..., keseringan lèk ndèk kono?",
          "Asline wong ngêndi ?",
          "iya [laugh] ngono (???) mba...",
          "Dadi têka kakèk'e ki jalur ya asli Jêmbêr."]:
    o = official_normalize(s)
    print("raw      : %s" % s)
    print("official : %s" % o)
    print("folded   : %s" % fold_diacritics(o))
    print("")

rec("scorer", version=scorer.VERSION, unit_checks=len(_checks),
    unit_checks_passed=sum(1 for _, c, _ in _checks if c),
    punct_removed=scorer.PUNCT_REMOVED,
    lowercases_sentence_initial=True, strips_periods_except_attached_ellipsis=True,
    does_not_strip_hyphens=True,
    path=str(out("scorer.py")))

## 3 — Segment table: timestamps, repair, drop, padding policy

The TSV carries `H:MM:SS` bounds with **one-second precision**. This cell parses them,
repairs the ones where a neighbouring row proves which of the two timestamps is the typo,
drops what cannot be repaired, and resolves overlaps by interval scheduling (earliest end
first) — which removes the fewest segments that makes each recording a clean partition.

It also measures the gap between consecutive segments, because that measurement decides
what padding can possibly do. Read the note under GAP STRUCTURE before touching `PAD_MODE`.

In [ ]:
# ============================================================
# CELL 3 - SEGMENT TABLE: timestamps, repair, drop, padding policy
# ============================================================

# ---- knobs -------------------------------------------------------------
# WRITE_AUDIO and DEBUG_N_RECORDINGS live in the RUN MODE block at the top of
# cell 1, because the output directory has to react to them. The smoke filter
# is applied HERE, at the source, so the whole pipeline stays self-consistent:
# a five-recording run produces a five-recording split, lexicon and manifest.

PAD_S    = 0.20             # frozen decision: 0.20 s of context on each side
PAD_MODE = "clip_to_neighbour"
# "clip_to_neighbour" - never cross into a neighbouring segment (frozen default)
# "allow_overlap"     - take the padding even when it duplicates neighbour audio
# Cell 7 measures what each mode would actually buy. Read that before flipping.

MAX_SEG_S = 30.0            # Whisper's window; anything longer must be split
MAX_REPAIR_SHIFT_S = 120.0  # refuse to "repair" a timestamp by more than this

head("LOAD JEMBER SEGMENT TABLE")
J = pd.read_csv(JEMBER_TSV, sep="\t", dtype=str, keep_default_na=False)
J.columns = [c.strip() for c in J.columns]
COL_REC = [c for c in J.columns if "audio" in c.lower()][0]
J = J.rename(columns={COL_REC: "audio_filename"})
for c in ("audio_filename", "start", "end", "text"):
    J[c] = J[c].astype(str).str.strip()
print("rows: %d   columns: %s" % (len(J), list(J.columns)))
print("recordings: %d" % J["audio_filename"].nunique())

if DEBUG_N_RECORDINGS:
    keep_ids = list(dict.fromkeys(J["audio_filename"]))[:int(DEBUG_N_RECORDINGS)]
    J = J[J["audio_filename"].isin(keep_ids)].copy()
    print("")
    print("!!! SMOKE RUN: restricted to %d recordings, %d rows." % (len(keep_ids), len(J)))
    print("!!! Proves the code runs. Proves nothing about the numbers.")
    print("!!! Set DEBUG_N_RECORDINGS = None in cell 1 for a real pass.")

def hms_to_s(x):
    # "H:MM:SS" in this corpus, but accept "MM:SS" and bare seconds too.
    parts = str(x).strip().split(":")
    try:
        vals = [float(p) for p in parts]
    except ValueError:
        return float("nan")
    s = 0.0
    for v in vals:
        s = s * 60 + v
    return s

J["start_raw"] = J["start"].map(hms_to_s)
J["end_raw"]   = J["end"].map(hms_to_s)
n_unparsed = int(J[["start_raw", "end_raw"]].isna().any(axis=1).sum())
print("unparseable timestamps: %d" % n_unparsed)

# Segment index = position in the ORIGINAL TSV order within its recording.
# Numbering happens BEFORE any repair or drop, so seg_id survives a change of
# drop policy: 63_002 is the same utterance no matter what we later decide.
J["seg_idx"] = J.groupby("audio_filename").cumcount() + 1
J["seg_id"]  = J["audio_filename"] + "_" + J["seg_idx"].map(lambda i: "%03d" % i)
J["row"]     = np.arange(len(J))
J["dur_raw"] = J["end_raw"] - J["start_raw"]

head("TIMESTAMP PRECISION AND GAP STRUCTURE")
# This is the measurement the padding decision hangs on. Do not skip it.
frac = np.concatenate([J["start_raw"].dropna().values % 1.0, J["end_raw"].dropna().values % 1.0])
print("timestamps that are whole seconds: %.1f %%" % (100.0 * float((frac == 0).mean())))
gaps = []
for _, g in J.groupby("audio_filename", sort=False):
    g = g.sort_values("row")
    gaps.extend((g["start_raw"].values[1:] - g["end_raw"].values[:-1]).tolist())
gaps = np.asarray([x for x in gaps if not math.isnan(x)], dtype=float)
n_gap_zero = int((gaps == 0).sum()); n_gap_pos = int((gaps > 0).sum()); n_gap_neg = int((gaps < 0).sum())
print("consecutive pairs: %d   gap == 0: %d (%.1f %%)   gap > 0: %d   gap < 0: %d"
      % (len(gaps), n_gap_zero, 100.0 * n_gap_zero / max(len(gaps), 1), n_gap_pos, n_gap_neg))
if n_gap_pos:
    desc(gaps[gaps > 0], "positive gaps")
print("")
print(">>> If every gap is 0 the annotation is a CONTIGUOUS partition: segment n")
print(">>> ends exactly where segment n+1 starts. Consequence for padding: under")
print(">>> PAD_MODE='clip_to_neighbour' there is nowhere to pad except at the")
print(">>> first and last segment of each recording. Cell 7 quantifies the cost")
print(">>> by measuring how much speech energy sits across those boundaries.")

# ---- repair pass -------------------------------------------------------
# Runs BEFORE the drop, and only where the evidence is unambiguous: one of the
# two timestamps agrees with the neighbouring row, so the other one is the typo.
head("REPAIR PASS (end <= start)")
J["start_s"] = J["start_raw"]; J["end_s"] = J["end_raw"]
J["repaired"] = ""
repairs = []
for fid, g in J.groupby("audio_filename", sort=False):
    idx = g.sort_values("row").index.tolist()
    for k, i in enumerate(idx):
        s, e = J.at[i, "start_raw"], J.at[i, "end_raw"]
        if math.isnan(s) or math.isnan(e) or e > s:
            continue
        prev_end   = J.at[idx[k - 1], "end_raw"] if k > 0 else 0.0
        next_start = J.at[idx[k + 1], "start_raw"] if k + 1 < len(idx) else float("nan")
        fix = None
        # (a) `end` agrees with the next row -> `start` is the typo. Restart the
        #     segment at the previous boundary, or at 0 if that is unusable too.
        if not math.isnan(next_start) and abs(e - next_start) < 1e-6:
            cand = prev_end if (prev_end <= e and e - prev_end <= MAX_REPAIR_SHIFT_S) else 0.0
            if cand < e and (e - cand) <= MAX_REPAIR_SHIFT_S:
                fix = (cand, e, "start<-%.0f" % cand)
        # (b) `start` agrees with the previous row -> `end` is the typo.
        if fix is None and k > 0 and abs(s - prev_end) < 1e-6 and not math.isnan(next_start) \
                and next_start > s and (next_start - s) <= MAX_REPAIR_SHIFT_S:
            fix = (s, next_start, "end<-%.0f" % next_start)
        if fix is None:
            continue
        # Collision guard: a repaired span may not swallow another segment.
        cs, ce, how = fix
        clash = [j for j in idx if j != i
                 and not math.isnan(J.at[j, "start_raw"])
                 and J.at[j, "end_raw"] > J.at[j, "start_raw"]
                 and min(ce, J.at[j, "end_raw"]) - max(cs, J.at[j, "start_raw"]) > 0.5]
        if clash:
            print("  %s: candidate %s rejected, would overlap %d other segment(s)"
                  % (J.at[i, "seg_id"], how, len(clash)))
            continue
        J.at[i, "start_s"] = cs; J.at[i, "end_s"] = ce; J.at[i, "repaired"] = how
        repairs.append(dict(seg_id=J.at[i, "seg_id"], raw="%.0f->%.0f" % (s, e),
                            fixed="%.0f->%.0f" % (cs, ce), how=how,
                            text=J.at[i, "text"][:60]))
if repairs:
    print(pd.DataFrame(repairs).to_string(index=False))
else:
    print("  nothing repairable")
print("repaired: %d" % len(repairs))

J["dur_s"] = J["end_s"] - J["start_s"]

# ---- drop pass ---------------------------------------------------------
head("DROP PASS")
J["drop_reason"] = ""
bad = J["dur_s"].isna() | (J["dur_s"] <= 0)
J.loc[bad, "drop_reason"] = "nonpositive_duration"
print("nonpositive / unparseable duration : %d" % int(bad.sum()))

# Overlaps. Keep the largest set of non-overlapping spans per recording
# (interval scheduling, earliest end first): deterministic, and it drops the
# minimum number of segments needed to make the recording a clean partition.
J["overlapped"] = False
n_overlap_drop = 0
for fid, g in J.groupby("audio_filename", sort=False):
    live = g[g["drop_reason"] == ""].sort_values(["end_s", "start_s", "row"])
    last_end = None
    for i, r in live.iterrows():
        if last_end is not None and r["start_s"] < last_end - 1e-9:
            J.at[i, "drop_reason"] = "overlap"
            n_overlap_drop += 1
        else:
            last_end = r["end_s"]
    # Flag every span - kept or dropped - that shared time with another one, so
    # a downstream notebook can exclude the survivors of a messy region too.
    spans = sorted(((J.at[i, "start_s"], J.at[i, "end_s"], i) for i in g.index
                    if not math.isnan(J.at[i, "start_s"]) and J.at[i, "end_s"] > J.at[i, "start_s"]))
    for a in range(len(spans)):
        s1, e1, i1 = spans[a]
        for b in range(a + 1, len(spans)):
            s2, e2, i2 = spans[b]
            if s2 >= e1:
                break
            if min(e1, e2) - max(s1, s2) > 0.5:
                J.at[i1, "overlapped"] = True
                J.at[i2, "overlapped"] = True
print("dropped for overlap                : %d" % n_overlap_drop)
print("segments touched by an overlap     : %d" % int(J["overlapped"].sum()))

too_long = (J["drop_reason"] == "") & (J["dur_s"] > MAX_SEG_S)
print("longer than %.0f s (kept, flagged)  : %d" % (MAX_SEG_S, int(too_long.sum())))

DROPPED = J[J["drop_reason"] != ""].copy()
S = J[J["drop_reason"] == ""].copy().reset_index(drop=True)
print("")
print("kept %d of %d segments  (%.2f h of %.2f h annotated)"
      % (len(S), len(J), S["dur_s"].sum() / 3600, J.loc[J["dur_raw"] > 0, "dur_raw"].sum() / 3600))
if len(DROPPED):
    print("")
    print(DROPPED.groupby("drop_reason").size().to_string())
    print("")
    print("dropped segment ids by recording:")
    for fid, g in DROPPED.groupby("audio_filename"):
        print("  %-5s %s" % (fid, ", ".join(g["seg_id"].tolist())))
    DROPPED[["seg_id", "audio_filename", "start", "end", "dur_raw", "drop_reason", "text"]] \
        .to_csv(out("dropped_segments.csv"), index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
    print("")
    print("saved ->", out("dropped_segments.csv"))

print("")
desc(S["dur_s"], "kept segment dur")
rec("segments",
    n_rows_tsv=len(J), n_recordings=int(J["audio_filename"].nunique()),
    n_kept=len(S), n_dropped=len(DROPPED), n_unparsed_timestamps=n_unparsed,
    drop_reasons={k: int(v) for k, v in DROPPED.groupby("drop_reason").size().items()},
    dropped_seg_ids=DROPPED["seg_id"].tolist()[:80],
    repairs=repairs, n_repaired=len(repairs),
    n_overlap_touched=int(J["overlapped"].sum()),
    gap_zero_pct=round(100.0 * n_gap_zero / max(len(gaps), 1), 2),
    gap_positive=n_gap_pos, gap_negative=n_gap_neg,
    whole_second_timestamp_pct=round(100.0 * float((frac == 0).mean()), 2),
    kept_duration=desc(S["dur_s"]), n_over_30s=int(too_long.sum()),
    pad_s=PAD_S, pad_mode=PAD_MODE, debug_n_recordings=DEBUG_N_RECORDINGS,
    write_audio=WRITE_AUDIO)

## 4 — Text: official normalization, folded view, full audit

`text_official` is the training target. `text_folded` is carried alongside it, never
instead of it, because the diacritic question is still open with the organiser.

The audit answers the questions the plan left hanging: how much token mass ends in a
period, how much is capitalised away from sentence start, which characters survive
normalization (that list is the CTC charset for notebook 05), and the full inventory of
words this corpus spells more than one way.

In [ ]:
# ============================================================
# CELL 4 - TEXT: official normalization, folded view, full audit
# ============================================================
head("NORMALIZE")
S["text_raw"]      = S["text"]
S["text_official"] = S["text_raw"].map(official_normalize)
S["text_folded"]   = S["text_official"].map(fold_diacritics)
S["n_words"]       = S["text_official"].map(lambda s: len(s.split()))
S["n_chars"]       = S["text_official"].map(len)

empty = S["text_official"].str.len() == 0
print("segments whose text is empty after normalization: %d" % int(empty.sum()))
if int(empty.sum()):
    print(S.loc[empty, ["seg_id", "text_raw"]].head(20).to_string(index=False))
    S = S[~empty].reset_index(drop=True)
    print("dropped them - a training pair with no target is not a training pair")

TOK  = [w for s in S["text_official"] for w in s.split()]     # scoring tokens
WORD = [w.lower() for s in S["text_official"] for w in re.findall(RE_TOKEN, s)]
print("scoring tokens: %d   word types (letters only, lowercased): %d"
      % (len(TOK), len(set(WORD))))

AUD = {}
def aud(k, v, note=""):
    AUD[k] = v
    if isinstance(v, (int, float)):
        pct = "  (%.1f %% of %d tokens)" % (100.0 * v / max(len(TOK), 1), len(TOK)) \
              if isinstance(v, int) and note == "tok" else ""
        print("  %-46s %s%s" % (k, v, pct))
    elif not k.endswith("_full") and k not in ("char_counts", "non_ascii_chars"):
        r = repr(v)
        print("  %-46s %s" % (k, r if len(r) <= 160 else r[:160] + " ..."))
    return v

head("AUDIT 1 - THE PERIOD DECISION")
# The organiser's punctuation list does not contain ".". Every token below is a
# token the model must learn to emit WITH its period or be scored wrong.
aud("tokens_total", len(TOK))
aud("tokens_ending_period", sum(1 for w in TOK if w.endswith(".")), "tok")
aud("tokens_containing_period", sum(1 for w in TOK if "." in w), "tok")
period_forms = collections.Counter(w for w in TOK if "." in w and not w.endswith("."))
aud("period_inside_forms", period_forms.most_common(20))
print("  most common sentence-final tokens:",
      collections.Counter(s.split()[-1] for s in S["text_official"] if s.split()).most_common(12))

head("AUDIT 2 - THE CASE DECISION")
# The list does not mention case folding either, so assume scoring is case
# sensitive. Sentence-initial capitals are learnable from position; the
# non-initial ones are the real bet - names, places, acronyms.
cap_tokens = [w for w in TOK if w[:1].isupper()]
noninit = [w for s in S["text_official"] for w in s.split()[1:] if w[:1].isupper()]
aud("tokens_capitalised", len(cap_tokens), "tok")
aud("tokens_capitalised_non_initial", len(noninit), "tok")
aud("top_non_initial_capitals", collections.Counter(noninit).most_common(30))
aud("tokens_all_caps", sum(1 for w in TOK if len(w) > 1 and w.isupper()), "tok")

head("AUDIT 3 - ELLIPSIS THAT SURVIVED")
ell = [w for w in TOK if "..." in w or "…" in w]
aud("tokens_with_attached_ellipsis", len(ell), "tok")
aud("attached_ellipsis_forms", collections.Counter(ell).most_common(40))
aud("segments_with_attached_ellipsis",
    int(S["text_official"].str.contains(r"\.\.\.", regex=True).sum()))
aud("tokens_with_two_dots", sum(1 for w in TOK if re.search(r"(?<!\.)\.\.(?!\.)", w)))
aud("unicode_ellipsis_u2026", sum(s.count("…") for s in S["text_official"]))
print("  NOTE: the normalizer only removes '...' with whitespace on both sides.")
print("  U+2026 is left alone on purpose - the published rule names three dots.")

head("AUDIT 4 - CHARACTER INVENTORY AFTER NORMALIZATION (the CTC charset)")
chars = collections.Counter(c for s in S["text_official"] for c in s)
aud("n_distinct_chars", len(chars))
aud("char_counts", dict(sorted(chars.items(), key=lambda kv: -kv[1])))
survivors = {c: n for c, n in chars.items() if not (c.isalnum() or c == " ")}
aud("non_alphanumeric_survivors", dict(sorted(survivors.items(), key=lambda kv: -kv[1])))
print("  survivors:", {("U+%04X %s" % (ord(c), c)): n
                       for c, n in sorted(survivors.items(), key=lambda kv: -kv[1])})
print("")
print("  >>> Curly quotes and U+2026 are NOT in the organiser's removal list, so")
print("  >>> they survive into the reference. Either the model learns to emit")
print("  >>> them, or they are permanent WER. Flag for the organiser question.")
nonascii = {c: n for c, n in chars.items() if ord(c) > 127}
aud("non_ascii_chars", {("U+%04X" % ord(c)): [c, n, unicodedata.name(c, "?")]
                        for c, n in sorted(nonascii.items(), key=lambda kv: -kv[1])})

head("AUDIT 5 - DIGITS")
dig = [w for w in TOK if any(ch.isdigit() for ch in w)]
aud("tokens_with_digit", len(dig), "tok")
aud("digit_forms", collections.Counter(dig).most_common(40))
print("  Digits are usually free WER: the model writes '5', the reference says")
print("  'lima', or the other way round. 63 tokens is small enough to ignore -")
print("  check the count before deciding to spend anything on it.")

head("AUDIT 6 - HYPHENS AND REDUPLICATION")
hyp = [w for w in TOK if "-" in w]
def is_redup(w):
    parts = fold_diacritics(w.lower().strip(".")).split("-")
    return len(parts) == 2 and parts[0] and parts[0] == parts[1]
redup = [w for w in hyp if is_redup(w)]
aud("tokens_with_hyphen", len(hyp), "tok")
aud("tokens_reduplication", len(redup), "tok")
aud("segments_with_hyphen", int(S["text_official"].str.contains("-").sum()))
aud("top_reduplications", collections.Counter(redup).most_common(30))
aud("hyphen_not_reduplication", collections.Counter(w for w in hyp if not is_redup(w)).most_common(30))

head("AUDIT 7 - DOUBLE SPELLINGS (the diacritic question, in full)")
# Group the lowercased vocabulary by its diacritic-folded form. Any group with
# more than one member is a word the corpus spells more than one way.
groups = collections.defaultdict(collections.Counter)
for w in WORD:
    groups[fold_diacritics(w)][w] += 1
ambig = {k: dict(v) for k, v in groups.items() if len(v) > 1}
ambig_mass = sum(sum(v.values()) for v in ambig.values())
aud("vocab_types_official", len(set(WORD)))
aud("vocab_types_folded", len(groups))
aud("types_recovered_by_folding", len(set(WORD)) - len(groups))
aud("n_ambiguous_forms", len(ambig))
aud("ambiguous_token_mass", ambig_mass)
aud("ambiguous_token_mass_pct", round(100.0 * ambig_mass / max(len(WORD), 1), 2))
aud("ambiguous_forms_full", {k: dict(sorted(v.items(), key=lambda kv: -kv[1]))
                             for k, v in sorted(ambig.items(),
                                                key=lambda kv: -sum(kv[1].values()))})
worst = sorted(ambig.items(), key=lambda kv: -sum(kv[1].values()))[:25]
print(pd.DataFrame([dict(folded=k, total=sum(v.values()),
                         variants=" | ".join("%s=%d" % (a, b)
                                             for a, b in sorted(v.items(), key=lambda kv: -kv[1])))
                    for k, v in worst]).to_string(index=False))
print("")
print("  >>> Folding recovers %d types and would make %.1f %% of the token mass"
      % (len(set(WORD)) - len(groups), 100.0 * ambig_mass / max(len(WORD), 1)))
print("  >>> spell consistently. That is the size of the prize - and the size of")
print("  >>> the loss if the organiser does NOT fold before scoring.")

head("AUDIT 8 - IDEMPOTENCY OF official_normalize")
viol = [s for s in S["text_raw"] if official_normalize(official_normalize(s)) != official_normalize(s)]
aud("non_idempotent_rows", len(viol))
aud("non_idempotent_examples", viol[:10])
if viol:
    print("  !! the rule order interacts with itself on these rows - inspect them")
else:
    print("  clean: applying the normalizer twice equals applying it once")

out("text_audit.json").write_text(
    json.dumps(jsonable(AUD), ensure_ascii=False, indent=1), encoding="utf-8")
print("")
print("saved ->", out("text_audit.json"))

rec("text_audit", **{k: v for k, v in jsonable(AUD).items()
                     if k not in ("char_counts", "ambiguous_forms_full", "non_ascii_chars")})
rec("text_audit", ambiguous_forms_top40={k: v for k, v in list(
        jsonable(AUD)["ambiguous_forms_full"].items())[:40]},
    charset="".join(sorted(chars)), non_ascii=list(jsonable(AUD)["non_ascii_chars"].keys()))

## 5 — Origin lexicon v1

Common Voice Indonesian attests Indonesian vocabulary; TTS Central Javanese attests
Javanese vocabulary; a marker list rescues the high-frequency Javanese function words that
attestation alone would misfile. Comparison happens on the diacritic-folded key.

This is a crude instrument on purpose. Its job is not linguistics, it is **consistency** —
so that "WER on Javanese words" means the same thing in notebooks 03 through 07. Frozen:
a better lexicon must be `lexicon_v2.csv`.

In [ ]:
# ============================================================
# CELL 5 - ORIGIN LEXICON v1  (frozen: a v2 must be a NEW file)
# ============================================================
# This is a crude instrument and it is meant to be. The Javanese-Indonesian
# lexical boundary is genuinely fuzzy - the organiser says so too. The point of
# the lexicon is not linguistics, it is CONSISTENCY: so that "WER on Javanese
# words" means the same thing in notebooks 03 through 07. `both` and `unk` are
# reported as their own classes and never folded into one of the two sides.

# High-frequency Javanese markers. Attestation alone cannot carry these: the
# words are so common in the Jember transcripts that a single Indonesian
# sentence containing one would otherwise flip them to `both`.
JV_MARKERS = """
sing iku ning iso lan ora nggo dadi kuwi karo ono wong okeh yen banget seko wes
dewe luwih mergo aku kowe arep gawe jare ngono ngene piye opo iki kono kene mau
mengko sopo pirang kabeh meneh ndue moco mbien ngeki liyo amergo menungso podo
arek dhewe mbek ndek terus
""".split()

def vocab_from_texts(texts, counter=None):
    # Word types for lexicon purposes: letters only, lowercased, diacritics
    # folded. Hyphenated forms are counted whole AND split, so that a
    # reduplication attested only as "wong-wong" still attests "wong".
    c = counter if counter is not None else collections.Counter()
    for t in texts:
        for w in re.findall(RE_TOKEN, str(t)):
            k = lex_key(w)
            if not k:
                continue
            c[k] += 1
            if "-" in k:
                for part in k.split("-"):
                    if part:
                        c[part] += 1
    return c

head("SOURCE VOCABULARIES")
CV_VOCAB = collections.Counter()
if CV_SENT is not None:
    cvdf = pd.read_csv(CV_SENT, sep="\t", dtype=str, keep_default_na=False,
                       quoting=csv.QUOTE_NONE, on_bad_lines="skip")
    col = "sentence" if "sentence" in cvdf.columns else cvdf.columns[1]
    vocab_from_texts(cvdf[col].tolist(), CV_VOCAB)
    print("Common Voice id : %-8d types   from %s (%d rows, column '%s')"
          % (len(CV_VOCAB), CV_SENT.name, len(cvdf), col))
else:
    print("Common Voice id : MISSING - every word will fall back to jv/unk")

TTS_VOCAB = collections.Counter()
n_tts_rows = 0
for mp in TTS_MAPPINGS:
    try:
        m = pd.read_csv(mp, sep="\t", dtype=str, keep_default_na=False)
    except Exception:
        continue
    col = "sentence" if "sentence" in m.columns else m.columns[-1]
    n_tts_rows += len(m)
    vocab_from_texts(m[col].tolist(), TTS_VOCAB)
print("Central JV TTS  : %-8d types   from %d mapping.tsv (%d rows)"
      % (len(TTS_VOCAB), len(TTS_MAPPINGS), n_tts_rows))

JEM_VOCAB = vocab_from_texts(S["text_official"].tolist())
print("Jember          : %-8d types   from %d segments" % (len(JEM_VOCAB), len(S)))

head("CLASSIFY")
# Comparison happens on the diacritic-folded key, so that "terus" (TTS) and
# "terus" (Jember, spelled with a caron) are recognised as the same word.
all_words = set(JEM_VOCAB) | set(CV_VOCAB) | set(TTS_VOCAB)
marker_keys = {lex_key(w) for w in JV_MARKERS}
rows = []
for w in sorted(all_words):
    in_cv, in_tts = w in CV_VOCAB, w in TTS_VOCAB
    if w in marker_keys:
        origin, source = "jv", "marker"
    elif in_cv and in_tts:
        origin, source = "both", "cv+tts"
    elif in_tts:
        origin, source = "jv", "tts"
    elif in_cv:
        origin, source = "id", "cv"
    else:
        origin, source = "unk", "jember-only"
    rows.append((w, origin, source))
LEX_DF = pd.DataFrame(rows, columns=["word", "origin", "source"])
LEX_DF.to_csv(out("lexicon_v1.csv"), index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
LEX = dict(zip(LEX_DF["word"], LEX_DF["origin"]))
print("lexicon_v1.csv : %d entries" % len(LEX_DF))
print(LEX_DF.groupby(["origin", "source"]).size().to_string())
print("")
print("saved ->", out("lexicon_v1.csv"))
print("")
print(">>> FROZEN. If a later notebook wants a better lexicon it writes")
print(">>> lexicon_v2.csv, so every number produced against v1 stays comparable.")

head("WHAT THE LEXICON SAYS ABOUT JEMBER TOKEN MASS")
mass = collections.Counter()
for w, n in JEM_VOCAB.items():
    mass[LEX.get(w, "unk")] += n
tot = sum(mass.values())
for o in ("jv", "id", "both", "unk"):
    print("  %-5s %8d tokens  %5.1f %%   %6d types"
          % (o, mass[o], 100.0 * mass[o] / max(tot, 1),
             sum(1 for w in JEM_VOCAB if LEX.get(w, "unk") == o)))
print("")
print("  most frequent unk (Javanese vocabulary neither corpus attests):")
print("   ", [w for w, _ in sorted(((w, n) for w, n in JEM_VOCAB.items()
                                    if LEX.get(w, "unk") == "unk"),
                                   key=lambda kv: -kv[1])[:30]])

head("PER-SEGMENT jv_ratio")
# Definition, frozen with the lexicon:
#     jv_ratio = n_jv / (n_jv + n_id)
# `both` and `unk` are deliberately outside the ratio. A segment with no
# jv and no id token gets NaN, which the split treats as its own stratum.
def classify_seg(text):
    c = collections.Counter()
    for w in text.split():
        k = lex_key(w)
        c[LEX.get(k, "unk") if k else "unk"] += 1
    return c

cls = S["text_official"].map(classify_seg)
for o in ("jv", "id", "both", "unk"):
    S["n_" + o] = cls.map(lambda c, o=o: c[o])
den = S["n_jv"] + S["n_id"]
S["jv_ratio"] = np.where(den > 0, S["n_jv"] / den.replace(0, np.nan), np.nan)
print("segments with no jv and no id token (jv_ratio = NaN): %d" % int(S["jv_ratio"].isna().sum()))
print(pd.cut(S["jv_ratio"], [-0.001, 0.001, 0.2, 0.4, 0.6, 0.8, 1.0],
             labels=["0 pure-id", "0-20%", "20-40%", "40-60%", "60-80%", "80-100%"])
      .value_counts().sort_index().to_string())
print("")
print("median jv_ratio: %.3f" % float(S["jv_ratio"].median()))

head("SEGMENT FLAGS")
S["flags"] = [
    ",".join([f for f, on in [
        ("zero_dur",  bool(r.repaired) and r.dur_raw <= 0),
        ("repaired",  bool(r.repaired)),
        ("overlap",   bool(r.overlapped)),
        ("has_ellipsis", "..." in r.text_official or "…" in r.text_official),
        ("has_redup", any(is_redup(w) for w in r.text_official.split() if "-" in w)),
        ("has_digit", any(ch.isdigit() for ch in r.text_official)),
    ] if on])
    for r in S.itertuples()
]
print(collections.Counter(f for fs in S["flags"] for f in fs.split(",") if f).most_common())

rec("lexicon",
    n_entries=len(LEX_DF), cv_types=len(CV_VOCAB), tts_types=len(TTS_VOCAB),
    jember_types=len(JEM_VOCAB), n_markers=len(marker_keys),
    by_origin={o: int((LEX_DF["origin"] == o).sum()) for o in ("jv", "id", "both", "unk")},
    jember_token_mass={o: int(mass[o]) for o in ("jv", "id", "both", "unk")},
    jember_token_mass_pct={o: round(100.0 * mass[o] / max(tot, 1), 2)
                           for o in ("jv", "id", "both", "unk")},
    jv_ratio_median=round(float(S["jv_ratio"].median()), 3),
    jv_ratio_nan_segments=int(S["jv_ratio"].isna().sum()),
    top_unk=[[w, int(n)] for w, n in sorted(((w, n) for w, n in JEM_VOCAB.items()
                                             if LEX.get(w, "unk") == "unk"),
                                            key=lambda kv: -kv[1])[:40]],
    flag_counts=dict(collections.Counter(f for fs in S["flags"] for f in fs.split(",") if f)),
    definition="jv_ratio = n_jv / (n_jv + n_id); both and unk excluded from the ratio")

## 6 — Frozen split

Grouped by **recording**, because the TSV has no speaker column and a segment-level random
split would put the same voices on both sides. Stratified on median `jv_ratio` × recording
duration, seeded, greedy bin-packing to a 15 % duration target.

The 5-fold column is not used today. It exists so that when a later notebook faces a
half-point WER difference, the answer is a five-fold re-run rather than an argument.

In [ ]:
# ============================================================
# CELL 6 - FROZEN SPLIT: grouped by recording, stratified, seeded
# ============================================================
# The Jember TSV has no speaker column. The only grouping unit available is the
# recording (session), so that is the unit. A segment-level random split would
# put the same two voices on both sides and hand back a WER that flatters the
# model by several points. Never do it, not even "just to check".

DEV_TARGET_FRAC = 0.15
N_FOLDS = 5

head("PER-RECORDING PROFILE")
R = S.groupby("audio_filename").agg(
        n_seg=("seg_id", "size"),
        dur=("dur_s", "sum"),
        n_words=("n_words", "sum"),
        jv=("jv_ratio", "median")).reset_index()
R["jv"] = R["jv"].fillna(R["jv"].median())
print("recordings: %d   total %.2f h   median %.1f min"
      % (len(R), R["dur"].sum() / 3600, R["dur"].median() / 60))
desc(R["dur"] / 60, "recording minutes")

# Two 3-way strata. Terciles rather than fixed cut points, so the buckets stay
# balanced whatever the corpus looks like.
def tercile(x, labels):
    try:
        return pd.qcut(x, 3, labels=labels, duplicates="drop")
    except ValueError:
        return pd.Series([labels[0]] * len(x), index=x.index)

R["jv_bin"]  = tercile(R["jv"].rank(method="first"),  ["jv_lo", "jv_mid", "jv_hi"])
R["dur_bin"] = tercile(R["dur"].rank(method="first"), ["d_short", "d_mid", "d_long"])
R["stratum"] = R["jv_bin"].astype(str) + "/" + R["dur_bin"].astype(str)
print("")
print(R.groupby("stratum").agg(n=("audio_filename", "size"), hours=("dur", lambda s: s.sum() / 3600))
      .to_string())

head("DEV SELECTION - greedy bin-packing inside each stratum")
# Inside a stratum: consider recordings longest first and take one only if it
# moves the accumulated duration CLOSER to that stratum's share of the target.
# Deterministic given SEED, and it never overshoots by a whole long recording.
rng = random.Random(SEED)
dev_ids = []
for st, g in R.groupby("stratum", sort=True):
    ids = g["audio_filename"].tolist()
    rng.shuffle(ids)                                    # tie-break, seeded
    dur = dict(zip(g["audio_filename"], g["dur"]))
    ids.sort(key=lambda i: -dur[i])                     # stable: ties keep the shuffle
    target = g["dur"].sum() * DEV_TARGET_FRAC
    acc, chosen = 0.0, []
    for i in ids:
        if abs(acc + dur[i] - target) < abs(acc - target):
            chosen.append(i); acc += dur[i]
    dev_ids.extend(chosen)
    print("  %-16s %2d/%2d recordings   %5.1f min of %6.1f min  (target %5.1f)"
          % (st, len(chosen), len(ids), acc / 60, g["dur"].sum() / 60, target / 60))

R["split"] = np.where(R["audio_filename"].isin(dev_ids), "dev", "train")
dev_frac = R.loc[R["split"] == "dev", "dur"].sum() / R["dur"].sum()
print("")
print("dev = %.2f %% of duration (target %.0f %%)" % (100 * dev_frac, 100 * DEV_TARGET_FRAC))
if abs(dev_frac - DEV_TARGET_FRAC) > 0.02:
    print("!! more than 2 points off target - inspect the stratum table above")

head("FOLDS - a grouped 5-fold over ALL recordings, for later")
# Not used by any notebook today. It exists so that when notebook 04 or 06 hits
# a difference of half a WER point, the answer is a re-run over five folds
# instead of an argument. Longest-processing-time scheduling inside each
# stratum keeps the folds close in duration.
fold_dur = [0.0] * N_FOLDS
R["fold"] = -1
for st, g in R.groupby("stratum", sort=True):
    ids = g.sort_values(["dur", "audio_filename"], ascending=[False, True])["audio_filename"].tolist()
    dur = dict(zip(g["audio_filename"], g["dur"]))
    for i in ids:
        f = int(np.argmin(fold_dur))
        R.loc[R["audio_filename"] == i, "fold"] = f
        fold_dur[f] += dur[i]
print("fold hours: " + "  ".join("f%d=%.2f" % (i, d / 3600) for i, d in enumerate(fold_dur)))
print("fold sizes: " + "  ".join("f%d=%d" % (i, int((R["fold"] == i).sum())) for i in range(N_FOLDS)))

SPLIT = R[["audio_filename", "split", "fold"]].copy()
SPLIT["_k"] = SPLIT["audio_filename"].map(lambda x: int(x) if str(x).isdigit() else 10 ** 9)
SPLIT = SPLIT.sort_values(["_k", "audio_filename"]).drop(columns="_k")
SPLIT.to_csv(out("split.csv"), index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
print("")
print("saved ->", out("split.csv"))
print(">>> FROZEN. split.csv goes into the repo and is never regenerated. Every")
print(">>> WER in notebooks 03-07 is measured against exactly these recordings.")

S = S.merge(R[["audio_filename", "split", "fold"]], on="audio_filename", how="left")

head("SPLIT REPORT")
rowsr = []
for sp in ("train", "dev"):
    sub = S[S["split"] == sp]
    rowsr.append(dict(split=sp, recordings=sub["audio_filename"].nunique(), segments=len(sub),
                      hours=round(sub["dur_s"].sum() / 3600, 3),
                      median_seg_s=round(float(sub["dur_s"].median()), 2),
                      p95_seg_s=round(float(sub["dur_s"].quantile(.95)), 2),
                      words=int(sub["n_words"].sum()),
                      jv_ratio_median=round(float(sub["jv_ratio"].median()), 3),
                      jv_ratio_p25=round(float(sub["jv_ratio"].quantile(.25)), 3),
                      jv_ratio_p75=round(float(sub["jv_ratio"].quantile(.75)), 3)))
print(pd.DataFrame(rowsr).to_string(index=False))

# Vocabulary the dev side alone carries. This is the floor on dev WER: a word
# the model never saw in training and cannot guess from context.
tr_vocab = collections.Counter()
dv_vocab = collections.Counter()
for t, sp in zip(S["text_official"], S["split"]):
    (tr_vocab if sp == "train" else dv_vocab).update(lex_key(w) for w in t.split() if lex_key(w))
dev_only = {w: n for w, n in dv_vocab.items() if w not in tr_vocab}
dev_only_mass = sum(dev_only.values())
print("")
print("dev-only word types : %d of %d dev types (%.1f %%)"
      % (len(dev_only), len(dv_vocab), 100.0 * len(dev_only) / max(len(dv_vocab), 1)))
print("dev-only token mass : %d of %d dev tokens (%.2f %%)  <- unavoidable dev WER floor"
      % (dev_only_mass, sum(dv_vocab.values()),
         100.0 * dev_only_mass / max(sum(dv_vocab.values()), 1)))
print("examples:", [w for w, _ in sorted(dev_only.items(), key=lambda kv: -kv[1])[:25]])

leak = set(S.loc[S["split"] == "train", "audio_filename"]) & set(S.loc[S["split"] == "dev", "audio_filename"])
print("")
print("recordings on both sides: %d  %s" % (len(leak), "(must be 0)" if not leak else sorted(leak)))
print("")
print(">>> %d recordings and %.2f h of dev means a WER difference under roughly"
      % (int((R['split'] == 'dev').sum()), S.loc[S['split'] == 'dev', 'dur_s'].sum() / 3600))
print(">>> one point is noise. When a decision turns on less than that, re-run it")
print(">>> over the 5 folds instead of arguing about the dev number.")

rec("split",
    seed=SEED, dev_target_frac=DEV_TARGET_FRAC, n_folds=N_FOLDS,
    grouping="audio_filename (recording/session) - no speaker column exists",
    strata="median jv_ratio tercile x recording duration tercile",
    dev_frac_duration=round(float(dev_frac), 4),
    dev_recordings=sorted(dev_ids, key=lambda x: (len(x), x)),
    per_split=jsonable(rowsr), fold_hours=[round(d / 3600, 3) for d in fold_dur],
    dev_only_types=len(dev_only), dev_only_token_mass=dev_only_mass,
    dev_only_token_pct=round(100.0 * dev_only_mass / max(sum(dv_vocab.values()), 1), 2),
    leak_recordings=sorted(leak))

## 7 — Segmentation: decode once, slice in memory, write FLAC

Each mp3 is decoded **exactly once** and then sliced as an array. Per-segment
`librosa`/`pydub` calls re-decode the whole recording ~32 times over and turn eight minutes
into several hours.

This cell also runs the boundary probe: for every segment boundary it measures whether the
0.2 s just outside it is closer to the recording's speech level or its noise floor. That
number is what the padding decision should actually rest on.

In [ ]:
# ============================================================
# CELL 7 - AUDIO: decode once per recording, write ONE 16 kHz FLAC per recording
# ============================================================
# Segments are NOT written as separate files. 6.638 loose clips is a bad way to
# ship 10 hours: Kaggle's dataset builder truncated it at 500 files, and even
# where it works, thousands of tiny files make every mount and every copy slow.
#
# Instead each recording is stored whole and the manifest carries the sample
# offsets. Notebook 03 reads a segment with
#     y, sr = sf.read(path, start=start_sample, stop=end_sample)
# which is a seek into the FLAC, not a re-decode. Same bytes on disk, 208 files
# instead of 6.638 - and the padding decision stops being baked into the audio,
# so it can be changed at training time without rebuilding anything.

import soundfile as sf
import librosa

TARGET_SR = 16000
FRAME = 320                       # 20 ms at 16 kHz, for the boundary probe

S = S.sort_values(["audio_filename", "row"]).reset_index(drop=True)

# Padding bounds. Under clip_to_neighbour a segment may never reach into a
# neighbouring segment's span; under allow_overlap it may, and the neighbour
# keeps its own audio regardless - the two reads simply share 0.2 s.
S["prev_end"] = S.groupby("audio_filename")["end_s"].shift(1)
S["next_start"] = S.groupby("audio_filename")["start_s"].shift(-1)

head("AUDIO PASS  (WRITE_AUDIO = %s)" % WRITE_AUDIO)
if not WRITE_AUDIO:
    print("Analysis pass: every mp3 is still decoded, every bound and every")
    print("measurement is computed, but no FLAC is written. Output stays a few MB.")
    print("Re-run with WRITE_AUDIO = True to produce the audio for notebook 03.")
    print("")
mp3s = {p.stem: p for p in sorted(JEMBER_AUDIO.glob("*.mp3"))}
need = list(dict.fromkeys(S["audio_filename"]))
print("recordings needed: %d   mp3 present: %d   missing: %s"
      % (len(need), sum(1 for f in need if f in mp3s), [f for f in need if f not in mp3s][:10]))

def frame_db(y):
    # RMS in dB over 20 ms frames. Used only for the boundary probe.
    n = (len(y) // FRAME) * FRAME
    if n == 0:
        return np.zeros(1)
    f = y[:n].reshape(-1, FRAME)
    return 20.0 * np.log10(np.sqrt((f ** 2).mean(axis=1)) + 1e-10)

results = {}
probe = dict(n=0, onset_active=0, offset_active=0, win_n=0, win_active=0)
PROBE_W = max(1, int(PAD_S * TARGET_SR) // FRAME)   # frames in one 0.2 s window
rec_bytes = {}
fail = []
t0 = time.time()
for k, fid in enumerate(need, 1):
    src = mp3s.get(fid)
    if src is None:
        fail.append((fid, "no mp3")); continue
    try:
        y, sr = librosa.load(str(src), sr=TARGET_SR, mono=True)
    except Exception as e:
        fail.append((fid, "%s: %s" % (type(e).__name__, str(e)[:60]))); continue
    n_samples = len(y)
    file_dur = n_samples / TARGET_SR
    db = frame_db(y)
    floor = float(np.percentile(db, 10)); speech = float(np.percentile(db, 90))
    thresh = (floor + speech) / 2.0                 # midpoint, in dB

    # BASE RATE. Without it the boundary number below means nothing: in a
    # conversation that is mostly speech, a window picked at random is above
    # threshold most of the time too. Same statistic, every 0.2 s window in the
    # recording, so the two are directly comparable.
    if db.size >= PROBE_W:
        _c = np.concatenate([[0.0], np.cumsum(db)])
        _win = (_c[PROBE_W:] - _c[:-PROBE_W]) / PROBE_W
        probe["win_n"] += int(_win.size)
        probe["win_active"] += int((_win > thresh).sum())

    rel = "recordings/%s.flac" % fid
    if WRITE_AUDIO:
        dst = AUDIO_DIR / ("%s.flac" % fid)
        pcm = np.clip(y, -1.0, 1.0)
        sf.write(str(dst), (pcm * 32767.0).astype(np.int16), TARGET_SR,
                 format="FLAC", subtype="PCM_16")
        rec_bytes[fid] = dst.stat().st_size

    g = S[S["audio_filename"] == fid]
    for i, r in g.iterrows():
        lo_lim = 0.0 if PAD_MODE == "allow_overlap" else (0.0 if math.isnan(r["prev_end"]) else r["prev_end"])
        hi_lim = file_dur if PAD_MODE == "allow_overlap" else (file_dur if math.isnan(r["next_start"]) else r["next_start"])
        s_pad = max(0.0, max(r["start_s"] - PAD_S, lo_lim))
        e_pad = min(file_dur, min(r["end_s"] + PAD_S, hi_lim))
        if e_pad <= s_pad:                          # nothing left after clipping
            s_pad, e_pad = max(0.0, r["start_s"]), min(file_dur, r["end_s"])
        a = int(round(s_pad * TARGET_SR)); b = int(round(e_pad * TARGET_SR))
        a = max(0, min(a, n_samples)); b = max(0, min(b, n_samples))
        if b - a < FRAME:
            # The span lies past the end of the mp3, or rounds to nothing. The
            # segment drops out of the manifest below.
            fail.append((r["seg_id"], "empty span %.2f-%.2f of %.2f s" % (s_pad, e_pad, file_dur)))
            continue
        # boundary probe, measured on the UNPADDED boundary either way
        for side, t in (("onset", r["start_s"]), ("offset", r["end_s"])):
            j0 = int(max(0.0, t - PAD_S if side == "onset" else t) * TARGET_SR) // FRAME
            j1 = j0 + PROBE_W
            w = db[j0:j1]
            if w.size:
                probe["n"] += 1
                if float(w.mean()) > thresh:
                    probe[side + "_active"] += 1
        results[r["seg_id"]] = dict(
            path=rel, start_pad=round(a / TARGET_SR, 4), end_pad=round(b / TARGET_SR, 4),
            dur_pad=round((b - a) / TARGET_SR, 4),
            start_sample=a, end_sample=b, file_dur=round(file_dur, 3),
            pad_clipped=bool((r["start_s"] - a / TARGET_SR) < PAD_S - 1e-6
                             or (b / TARGET_SR - r["end_s"]) < PAD_S - 1e-6),
            end_beyond_file=bool(r["end_s"] > file_dur + 0.5))
    if k % 25 == 0 or k == len(need):
        el = time.time() - t0
        print("  %3d/%3d recordings  %5d segments  %6.1f s elapsed  eta %5.1f s"
              % (k, len(need), len(results), el, el / k * (len(need) - k)))

print("")
print("%s %d recordings, %d segments indexed, in %.1f s"
      % ("wrote" if WRITE_AUDIO else "measured", len(need) - len(set(f for f, _ in fail)),
         len(results), time.time() - t0))
if fail:
    print("!! problems:", fail[:10])
if WRITE_AUDIO:
    tot_mb = sum(rec_bytes.values()) / 1e6
    sz = sorted(rec_bytes.values())
    print("recordings/ : %d flac, %.0f MB total, smallest %.2f MB, largest %.2f MB"
          % (len(rec_bytes), tot_mb, sz[0] / 1e6, sz[-1] / 1e6))

A = pd.DataFrame.from_dict(results, orient="index")
A.index.name = "seg_id"
S = S.merge(A.reset_index(), on="seg_id", how="left")
missing_audio = S["path"].isna()
if int(missing_audio.sum()):
    print("!! %d segments have no audio span - dropping them from the manifest"
          % int(missing_audio.sum()))
    S = S[~missing_audio].reset_index(drop=True)

head("PADDING - WHAT IT ACTUALLY BOUGHT")
gained = (S["dur_pad"] - S["dur_s"]).clip(lower=0)
print("segments whose padding was clipped by a neighbour or the file edge: %d of %d (%.1f %%)"
      % (int(S["pad_clipped"].sum()), len(S), 100.0 * float(S["pad_clipped"].mean())))
print("mean context actually gained: %.3f s of the %.2f s requested (%.0f %% of the ask)"
      % (gained.mean(), 2 * PAD_S, 100.0 * gained.mean() / max(2 * PAD_S, 1e-9)))
print("segments whose annotated end runs past the end of the mp3: %d"
      % int(S["end_beyond_file"].sum()))
print("")
print(">>> Because the audio is stored per recording, these bounds are just two")
print(">>> numbers in the manifest. Changing PAD_MODE later costs a recompute of")
print(">>> start_sample/end_sample - not a rebuild of the dataset.")

head("BOUNDARY PROBE - IS THE 1-SECOND TIMESTAMP CUTTING WORDS IN HALF?")
n_pr = max(probe["n"], 1)
act = probe["onset_active"] + probe["offset_active"]
base = 100.0 * probe["win_active"] / max(probe["win_n"], 1)
bnd = 100.0 * act / n_pr
print("boundaries probed         : %d" % probe["n"])
print("onset windows with speech : %d" % probe["onset_active"])
print("offset windows with speech: %d" % probe["offset_active"])
print("")
print("  just outside a boundary : %.1f %% of windows are above the speech threshold" % bnd)
print("  anywhere in the audio   : %.1f %% (base rate, same statistic, %d windows)"
      % (base, probe["win_n"]))
print("  ratio                   : %.2f x base" % (bnd / max(base, 1e-9)))
print("")
if bnd < base * 0.7:
    print(">>> Boundaries sit in speech LESS often than a random moment does, so the")
    print(">>> annotators were placing them at pauses. The 1-second grid is coarse,")
    print(">>> but it is not landing mid-word as often as feared - padding buys less")
    print(">>> than the raw number suggests.")
elif bnd > base * 1.1:
    print(">>> Boundaries sit in speech MORE often than a random moment does. The")
    print(">>> 1-second grid is actively cutting through words, and clip_to_neighbour")
    print(">>> leaves every one of those cuts unrepaired.")
else:
    print(">>> Boundaries are indistinguishable from random moments: the annotation")
    print(">>> grid ignores where the speech actually is. Roughly that share of clips")
    print(">>> starts or ends mid-word, and clip_to_neighbour repairs none of them.")
print("")

desc(S["dur_pad"], "segment duration")
rec("audio",
    storage="one 16 kHz mono FLAC per recording; segments addressed by sample offset",
    target_sr=TARGET_SR, n_recordings=len(rec_bytes) if WRITE_AUDIO else len(need),
    n_segments=len(S), n_failed=len(fail), failed=[list(x) for x in fail[:20]],
    seconds_elapsed=round(time.time() - t0, 1),
    bytes_total=int(sum(rec_bytes.values())), mb_total=round(sum(rec_bytes.values()) / 1e6, 1),
    pad_s=PAD_S, pad_mode=PAD_MODE,
    n_pad_clipped=int(S["pad_clipped"].sum()),
    pad_clipped_pct=round(100.0 * float(S["pad_clipped"].mean()), 2),
    mean_context_gained_s=round(float(gained.mean()), 4),
    n_end_beyond_file=int(S["end_beyond_file"].sum()),
    boundary_probe_n=probe["n"], boundary_in_speech_pct=round(bnd, 2),
    speech_base_rate_pct=round(base, 2), boundary_vs_base_ratio=round(bnd / max(base, 1e-9), 3),
    probe_windows=probe["win_n"],
    boundary_onset_in_speech_pct=round(100.0 * probe["onset_active"] / max(n_pr / 2, 1), 2),
    boundary_offset_in_speech_pct=round(100.0 * probe["offset_active"] / max(n_pr / 2, 1), 2),
    segment_duration=desc(S["dur_pad"]))

## 8 — Manifest

`start_s`/`end_s`/`dur_s` describe the file on disk (padded). The annotated bounds are kept
as `ann_start_s`/`ann_end_s` so the padding decision can be revisited without re-reading the
TSV. Written with `QUOTE_ALL` — the transcripts contain commas, colons and quotes.

In [ ]:
# ============================================================
# CELL 8 - MANIFEST
# ============================================================
# `path` points at the RECORDING, not at a per-segment file. Read one segment
# with sf.read(path, start=start_sample, stop=end_sample) - an exact integer
# seek, no float rounding, no re-decode.
#
# start_s / end_s / dur_s are the padded bounds those samples correspond to.
# The annotated bounds stay in ann_start_s / ann_end_s, so the padding decision
# can be recomputed at training time without touching the audio at all.
head("MANIFEST")
S["pad_clipped_flag"] = S["pad_clipped"].fillna(False).astype(bool)
S["flags"] = [
    ",".join([f for f in (fl.split(",") if fl else []) if f] + (["pad_clipped"] if pc else []))
    for fl, pc in zip(S["flags"], S["pad_clipped_flag"])
]

M = pd.DataFrame(dict(
    seg_id=S["seg_id"],
    audio_filename=S["audio_filename"],
    path=S["path"],
    start_sample=S["start_sample"].astype("int64"),
    end_sample=S["end_sample"].astype("int64"),
    start_s=S["start_pad"].round(4),
    end_s=S["end_pad"].round(4),
    dur_s=S["dur_pad"].round(4),
    ann_start_s=S["start_s"].round(3),
    ann_end_s=S["end_s"].round(3),
    text_raw=S["text_raw"],
    text_official=S["text_official"],
    text_folded=S["text_folded"],
    n_words=S["n_words"].astype(int),
    n_chars=S["n_chars"].astype(int),
    jv_ratio=S["jv_ratio"].round(4),
    n_jv=S["n_jv"].astype(int),
    n_id=S["n_id"].astype(int),
    n_both=S["n_both"].astype(int),
    n_unk=S["n_unk"].astype(int),
    split=S["split"],
    fold=S["fold"].astype(int),
    flags=S["flags"],
))
M["_k"] = M["audio_filename"].map(lambda x: int(x) if str(x).isdigit() else 10 ** 9)
M = M.sort_values(["_k", "audio_filename", "seg_id"]).drop(columns="_k").reset_index(drop=True)

# QUOTE_ALL is not cosmetic: the transcripts contain commas, colons and quotes.
M.to_csv(out("manifest.csv"), index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
print("manifest.csv : %d rows x %d columns" % M.shape)
print("saved ->", out("manifest.csv"))
print("")
print(M.head(8).to_string(index=False, max_colwidth=34))
print("")
print("columns:", list(M.columns))
print("")
print("HOW NOTEBOOK 03+ READS ONE SEGMENT:")
print("    import soundfile as sf")
print("    r = manifest.iloc[i]")
print("    y, sr = sf.read(ROOT / r.path, start=r.start_sample, stop=r.end_sample,")
print("                    dtype='float32')      # 16 kHz mono, seek not decode")

rec("manifest", n_rows=len(M), columns=list(M.columns),
    hours=round(float(M["dur_s"].sum()) / 3600, 3),
    words=int(M["n_words"].sum()),
    by_split={k: int(v) for k, v in M.groupby("split").size().items()},
    hours_by_split={k: round(float(v) / 3600, 3)
                    for k, v in M.groupby("split")["dur_s"].sum().items()})

## 9 — Self-validation

Each check is a way this notebook could have failed silently. A green board is what
"Gate 1 passed" means; anything red blocks notebooks 03–05.

The demo at the end prices the cheapest possible mistake: a hypothesis that only lowercases,
strips periods and folds diacritics. No acoustic model is involved, and the WER is not small.

> **Correction 2026-08-25.** The scorer was fixed the same day: the real normalizer *does*
> lowercase (sentence-initial only, not everywhere) and *does* strip periods. Two of this
> demo's three "mistakes" are therefore partly what the official scorer itself does before
> comparing - only full-string lowercasing and diacritic folding remain genuine mistakes.
> The WER number below predates the fix; re-run this cell before quoting it again.

In [ ]:
# ============================================================
# CELL 9 - SELF-VALIDATION
# ============================================================
# Every check below is a way this notebook could have failed silently. A green
# board here is what "Gate 1 passed" means; anything red blocks notebooks 03+.
head("SELF-VALIDATION")
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:160]))
    print("  [%s] %-52s %s" % ("PASS" if cond else "FAIL", name, detail))
    return bool(cond)

flacs = sorted(AUDIO_DIR.glob("*.flac"))
if WRITE_AUDIO:
    sizes = [p.stat().st_size for p in flacs]
    check("one flac per recording", len(flacs) == M["audio_filename"].nunique(),
          "%d flac / %d recordings" % (len(flacs), M["audio_filename"].nunique()))
    check("no zero-byte flac", all(z > 0 for z in sizes),
          "smallest %.2f MB" % (min(sizes) / 1e6 if sizes else -1))
    check("every manifest path exists on disk",
          all((OUT_DIR / p).exists() for p in M["path"].unique()),
          "%d distinct paths" % M["path"].nunique())
    over = [p.name for p in flacs
            if sf.info(str(p)).frames < int(M.loc[M["audio_filename"] == p.stem,
                                                  "end_sample"].max())]
    check("no segment reaches past the end of its recording", not over, over[:5])
else:
    print("  [skip] the four on-disk audio checks - WRITE_AUDIO is False.")
    print("         Bounds, sample offsets and the boundary probe were still")
    print("         measured from the decoded audio; only the files were not written.")

tsv_h = float(S["dur_s"].sum()) / 3600            # annotated, after repair+drop
man_h = float(M["ann_end_s"].sub(M["ann_start_s"]).sum()) / 3600
check("manifest duration matches TSV after drops (<1 %)",
      abs(man_h - tsv_h) / max(tsv_h, 1e-9) < 0.01,
      "%.4f h vs %.4f h" % (man_h, tsv_h))

both_sides = set(M.loc[M["split"] == "train", "audio_filename"]) & \
             set(M.loc[M["split"] == "dev", "audio_filename"])
check("no recording in both train and dev", not both_sides, sorted(both_sides)[:5])

check("official_normalize is idempotent on the corpus",
      all(official_normalize(t) == t for t in M["text_official"]),
      "%d rows" % len(M))

refs = M["text_official"].tolist()
w0 = scorer.wer(refs, refs)["wer"]
w1 = scorer.wer(refs, [""] * len(refs))["wer"]
check("scorer.wer(refs, refs) == 0.0", w0 == 0.0, "%.6f" % w0)
check("scorer.wer(refs, empty) == 1.0", w1 == 1.0, "%.6f" % w1)

ok_all, checks_scorer = scorer.self_test(refs=refs, verbose=False)
check("scorer.self_test on the real references", ok_all,
      "%d/%d" % (sum(1 for _, c, _ in checks_scorer if c), len(checks_scorer)))
for nm, cond, det in checks_scorer:
    if not cond:
        print("      FAILED sub-check:", nm, det)

if WRITE_AUDIO and flacs:
    bad_sr = [p.name for p in flacs if sf.info(str(p)).samplerate != TARGET_SR]
    bad_ch = [p.name for p in flacs if sf.info(str(p)).channels != 1]
    check("every recording is 16 kHz", not bad_sr, bad_sr[:5])
    check("every recording is mono", not bad_ch, bad_ch[:5])
    # Round-trip through the exact call notebook 03 will make.
    rs = random.Random(SEED)
    rows = M.sample(n=min(30, len(M)), random_state=SEED)
    bad_len, silent = [], []
    for _, r in rows.iterrows():
        y, sr = sf.read(str(OUT_DIR / r["path"]), start=int(r["start_sample"]),
                        stop=int(r["end_sample"]), dtype="float32")
        if sr != TARGET_SR or y.ndim != 1 or len(y) != int(r["end_sample"]) - int(r["start_sample"]):
            bad_len.append(r["seg_id"])
        elif float(np.sqrt((y.astype(np.float64) ** 2).mean())) < 1e-5:
            silent.append(r["seg_id"])
    check("round-trip: 30 segments read back at the right length", not bad_len, bad_len[:5])
    check("round-trip: none of them is digital silence", not silent, silent[:5])
else:
    print("  [skip] the four round-trip checks - no audio on disk in this pass")

check("every clip is shorter than %.0f s" % MAX_SEG_S, bool((M["dur_s"] < MAX_SEG_S).all()),
      "longest %.2f s" % float(M["dur_s"].max()))
check("no empty transcript", bool((M["n_words"] > 0).all()),
      "min %d words" % int(M["n_words"].min()))
check("seg_id unique", M["seg_id"].is_unique, "%d ids" % M["seg_id"].nunique())
check("split.csv covers every manifest recording",
      set(M["audio_filename"]) <= set(SPLIT["audio_filename"]),
      "%d vs %d" % (M["audio_filename"].nunique(), len(SPLIT)))
check("not a smoke run", DEBUG_N_RECORDINGS is None,
      "DEBUG_N_RECORDINGS = %s%s" % (DEBUG_N_RECORDINGS,
       " - expected to FAIL in phase 0" if DEBUG_N_RECORDINGS else ""))

n_pass = sum(1 for c in CHECKS if c["passed"])
gate1 = bool(n_pass == len(CHECKS) and WRITE_AUDIO and DEBUG_N_RECORDINGS is None)
print("")
print("%d/%d checks passed" % (n_pass, len(CHECKS)))
smoke_only = (DEBUG_N_RECORDINGS is not None
              and n_pass == len(CHECKS) - 1
              and not [c for c in CHECKS if not c["passed"] and c["check"] != "not a smoke run"])
if smoke_only:
    print(">>> SMOKE RUN OK. The only failure is the smoke flag itself, which is")
    print(">>> supposed to fail here. The pipeline executes end to end - now set")
    print(">>> DEBUG_N_RECORDINGS = None in cell 1 and run phase 1 for real.")
elif n_pass != len(CHECKS):
    print(">>> BLOCKED - fix the FAIL rows above before any GPU lane starts.")
elif not WRITE_AUDIO:
    print(">>> ANALYSIS PASS OK. Every number here is final and can be reviewed,")
    print(">>> but Gate 1 is not passed until the clips exist: re-run with")
    print(">>> WRITE_AUDIO = True once the numbers have been agreed.")
else:
    print(">>> GATE 1 PASSED. Notebooks 03, 04, 05 may start.")

head("SCORER DEMO - what notebooks 03-07 will call")
# A synthetic hypothesis: lowercase everything, drop the periods, fold the
# diacritics. That is exactly the "helpfully cleaned" output a naive pipeline
# produces, and the table below is what it costs.
demo_ref = M["text_official"].head(400).tolist()
demo_hyp = [fold_diacritics(t.lower().replace(".", "")) for t in demo_ref]
d = scorer.wer_by_origin(demo_ref, demo_hyp, LEX)
print("A hypothesis that only lowercases, strips periods and folds diacritics:")
print("  wer_total %.4f   jv %.4f (%d ref tok)   id %.4f (%d)   both %.4f (%d)   unk %.4f (%d)"
      % (d["wer_total"], d["wer_jv"], d["n_ref_jv"], d["wer_id"], d["n_ref_id"],
         d["wer_both"], d["n_ref_both"], d["wer_unk"], d["n_ref_unk"]))
print("  sub %.4f  del %.4f  ins %.4f   cosmetic substitutions: %.1f %%"
      % (d["sub_rate"], d["del_rate"], d["ins_rate"], d["cosmetic_sub_pct"]))
print("")
print(">>> That WER is entirely self-inflicted: no acoustic model was involved.")
print(">>> It is the cost of training on 'cleaned' text instead of the")
print(">>> normalizer's own output.")
print("")
et = scorer.error_table(demo_ref, demo_hyp, LEX, top_k=15)
print(et.to_string(index=False) if hasattr(et, "to_string") else et)

rec("validation", checks=CHECKS, n_passed=n_pass, n_total=len(CHECKS),
    write_audio=WRITE_AUDIO, gate1_passed=gate1,
    cleaned_text_self_inflicted_wer=round(d["wer_total"], 4),
    cleaned_text_cosmetic_sub_pct=d["cosmetic_sub_pct"])

## 10 — Digest and packaging

Run last. Copy the marked block back into the chat.

In [ ]:
# ============================================================
# CELL 10 - DIGEST + PACKAGING
# Run this last. Copy the ENTIRE block between the BEGIN/END markers and paste
# it back into the Claude conversation. That is the handoff.
# ============================================================
head("ARTIFACT FINGERPRINTS")
# The two-pass workflow rests on one claim: pass 2 reproduces pass 1's small
# artifacts exactly, and only adds audio. That claim is cheap to VERIFY and
# expensive to merely believe - a silent drift would mean the numbers that were
# reviewed are not the numbers that shipped. So hash them and carry the hashes
# in the digest, where the two passes can be compared line by line.
import hashlib
FP = {}
for name in ("manifest.csv", "split.csv", "lexicon_v1.csv", "scorer.py",
             "text_audit.json", "dropped_segments.csv"):
    p = OUT_DIR / name
    if p.exists():
        FP[name] = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
        print("  %-24s %s   %8.1f KB" % (name, FP[name], p.stat().st_size / 1024))
print("")
if WRITE_AUDIO:
    print(">>> These must match the hashes printed by the analysis pass. If one")
    print(">>> moved, something non-deterministic crept in (a library version, an")
    print(">>> unseeded shuffle) and the artifacts need re-reviewing, not shipping.")
else:
    print(">>> Write these down. Pass 2 prints the same list, and every line must")
    print(">>> be identical - that is what makes reviewing pass 1 meaningful.")
rec("fingerprints", **FP)

head("PACKAGE recordings/ AS A SELF-CONTAINED KAGGLE DATASET")
if WRITE_AUDIO:
    # Notebooks 03+ mount ONE dataset and get everything: the audio, the
    # manifest, the frozen split, the lexicon and the scorer. Without this copy
    # they would have to mount the audio and paste the small files in by hand.
    for name in ("manifest.csv", "split.csv", "lexicon_v1.csv", "scorer.py",
                 "text_audit.json", "dropped_segments.csv"):
        src = OUT_DIR / name
        if src.exists():
            shutil.copy2(str(src), str(META_DIR / name))
    print("recordings/_meta/ :", sorted(p.name for p in META_DIR.iterdir()))
    n_flac = len(list(AUDIO_DIR.glob("*.flac")))
    seg_mb = sum(p.stat().st_size for p in AUDIO_DIR.rglob("*")) / 1e6
    print("recordings/       : %d flac (one per recording), %.0f MB total" % (n_flac, seg_mb))
else:
    n_flac, seg_mb = 0, 0.0
    try:
        shutil.rmtree(str(AUDIO_DIR))
    except Exception:
        pass
    print("ANALYSIS PASS - no audio written, recordings/ removed from the output so")
    print("the whole archive stays small enough to download in one piece.")

EXPECTED = ["env", "scorer", "segments", "text_audit", "lexicon", "split",
            "audio", "manifest", "validation", "fingerprints"]
missing_sections = [k for k in EXPECTED if k not in FINDINGS]
if missing_sections:
    print("")
    print("!" * 72)
    print("INCOMPLETE DIGEST - missing sections: %s" % missing_sections)
    print("Cause is almost always a partial re-run: cell 1 resets FINDINGS, so")
    print("any cell not executed after it contributes nothing. FIX: Run All.")
    print("!" * 72)

FINDINGS["_meta"] = dict(
    notebook="mozilla-javanese data build + eval harness v1",
    notebook_id=NOTEBOOK_ID,
    generated_for="handoff to Claude - do not edit by hand",
    complete=(not missing_sections), missing_sections=missing_sections,
    artifacts=sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))

blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False,
                  separators=(",", ":"), sort_keys=True)
print("")
print("===== BEGIN CLAUDE-HANDOFF-JSON =====")
print(blob)
print("===== END CLAUDE-HANDOFF-JSON =====")
print("")
print("size: %.1f KB - select the whole block above and paste it into the chat" % (len(blob) / 1024))

try:
    out("dataset_findings.json").write_text(
        json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1, sort_keys=True),
        encoding="utf-8")
    print("saved ->", out("dataset_findings.json"))
    if WRITE_AUDIO:
        # The digest carries the fingerprints, so it belongs INSIDE the
        # published dataset - otherwise phase 3 has nothing to compare against.
        shutil.copy2(str(out("dataset_findings.json")),
                     str(META_DIR / "dataset_findings.json"))
        print("copied -> recordings/_meta/dataset_findings.json")
    sys.stdout.flush()
    print("saved ->", LOG_PATH)
except Exception as e:
    print("could not save:", e)

# Sweep anything Kaggle dropped loose in /kaggle/working into _kaggle-artifacts,
# so the downloaded archive contains only the mirrored tree. Files Kaggle writes
# at save time (after this cell) will still land loose - move those by hand.
try:
    moved = []
    for q in WORKING.iterdir():
        if q.is_file():
            q.replace(ARTIFACT_DIR / q.name); moved.append(q.name)
    if moved:
        print("swept into _kaggle-artifacts:", moved)
except Exception as e:
    print("sweep skipped:", e)

head("LITE ARCHIVE - THE ONLY THING THAT NEEDS TO BE DOWNLOADED")
# "Download all" means dragging the audio across the wire too, and half a
# gigabyte over a flaky connection fails more often than it finishes. The
# analysis artifacts are three orders of magnitude smaller, so they get their
# own archive that can be pulled on its own from the Output file browser.
import zipfile
LITE = OUT_DIR / ("%s-lite.zip" % NOTEBOOK_ID)
if LITE.exists():
    LITE.unlink()
lite_files = [p for p in (OUT_DIR / n for n in
                          ("dataset_findings.json", "text_audit.json", "manifest.csv",
                           "split.csv", "lexicon_v1.csv", "scorer.py",
                           "dropped_segments.csv", "run-log.txt"))
              if p.exists()]
sys.stdout.flush()          # so run-log.txt goes in as complete as it can be
with zipfile.ZipFile(LITE, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for p in lite_files:
        z.write(str(p), "findings/%s/%s" % (NOTEBOOK_ID, p.name))
for p in lite_files:
    print("    %-24s %8.1f KB" % (p.name, p.stat().st_size / 1024))
print("")
print("%s : %.1f MB   vs %.0f MB for the full output"
      % (LITE.name, LITE.stat().st_size / 1e6, seg_mb + LITE.stat().st_size / 1e6))

head("NEXT STEPS, BY HAND")
if DEBUG_N_RECORDINGS:
    print("THIS WAS PHASE 0 - A SMOKE RUN ON %d RECORDINGS." % DEBUG_N_RECORDINGS)
    print("")
    print("It answers one question only: does the notebook run end to end without")
    print("throwing? Every number above is built from a sliver of the corpus and")
    print("means nothing. The split, the lexicon and the manifest are all wrong by")
    print("construction, which is why they were written to findings/%s%s/"
          % (NOTEBOOK_ID, RUN_TAG))
    print("rather than the real folder. Do not download, commit or publish any of it.")
    print("")
    print("If the board in cell 9 showed no FAIL other than 'not a dry run':")
    print("  -> set DEBUG_N_RECORDINGS = None in cell 1 and Run All. That is phase 1.")
elif not WRITE_AUDIO:
    print("THIS WAS PASS 1 OF 2 - analysis only.")
    print("")
    print("1. Output tab -> Download all. The entire output is a few MB in this")
    print("   pass, so there is no large file to fail halfway. (%s is" % LITE.name)
    print("   in there too if you would rather grab a single file.)")
    print("")
    print("2. Unzip onto the repo root - it already contains findings/%s/." % NOTEBOOK_ID)
    print("")
    print("3. Tell Claude it has landed, or paste the CLAUDE-HANDOFF-JSON block")
    print("   above. Review the numbers - especially the boundary probe, which")
    print("   decides PAD_MODE - before spending a run on the audio.")
    print("")
    print("4. THEN set WRITE_AUDIO = True in cell 3 and Run All again. Same seed,")
    print("   same rules, so manifest.csv / split.csv / lexicon_v1.csv come out")
    print("   identical - pass 2 only adds the clips.")
else:
    print("THIS WAS PASS 2 - the clips exist.")
    print("")
    print("1. Output tab -> click %s -> download that ONE file." % LITE.name)
    print("   Do NOT use 'Download all': that drags the %.0f MB of audio with it." % seg_mb)
    print("   Unzip onto the repo root - it already contains findings/%s/." % NOTEBOOK_ID)
    print("")
    print("2. Publish the audio WITHOUT downloading it. On this notebook's Output")
    print("   tab use 'New Dataset' (create a dataset from the notebook output)")
    print("   and name it 'jember-segments-16k'. Kaggle builds it server-side, so")
    print("   nothing crosses your connection. Keep recordings/_meta/ inside it.")
    print("")
    print("3. Commit split.csv, lexicon_v1.csv, scorer.py, manifest.csv to the repo.")
    print("")
    print("4. Paste the CLAUDE-HANDOFF-JSON block above into the chat. If even the")
    print("   lite zip refuses to download, that block ALONE is enough to review")
    print("   this run - it carries every number the notebook measured.")

head("DECISIONS THIS NOTEBOOK MEASURED BUT DID NOT TAKE")
for q in [
    "Padding: the annotation is contiguous, so clip_to_neighbour pads almost",
    "  nothing. The boundary probe in cell 7 says how much speech that leaves",
    "  outside the clips. Flipping PAD_MODE to 'allow_overlap' is a one-line",
    "  change plus a re-run - decide before the segments dataset is published.",
    "Diacritics: text_folded is carried but never used as the training target.",
    "  The organiser has not said whether scoring folds. Until they do, train on",
    "  text_official and keep the folded column as the fallback experiment.",
    "Curly quotes and U+2026 survive the official normalizer. Either the model",
    "  learns to emit them or they are permanent WER. Worth one question.",
    "Case and periods are kept, on the reading that the published normalizer",
    "  lists neither. The demo in cell 9 prices what being wrong about that",
    "  would cost - it is not small.",
    "Common Voice and TTS Central Javanese are used here ONLY as vocabulary",
    "  attestation. Whether to train on them is notebook 06's question.",
]:
    print(" -", q)